In [70]:
# Clone your GitHub repo (you’ll be prompted to authorize if it's private)
!git clone https://github.com/colterwood/LHL-final-final-project.git

fatal: destination path 'LHL-final-final-project' already exists and is not an empty directory.


In [2]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup, Comment
import requests
from io import StringIO
import string
import time
import re
import os

In [ ]:
year = 2024
url = f"https://www.basketball-reference.com/wnba/years/{year}.html"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")
comments = soup.find_all(string=lambda text: isinstance(text, Comment))

In [ ]:
# full list of table IDs to load from the team page
table_ids = [
    "per_game-team", "per_game-opponent",
    "totals-team", "totals-opponent",
    "advanced-team", "per_poss-team", "per_poss-opponent",
    "shooting-team", "shooting-opponent"
]

# columns to exclude from prefixing
no_prefix_cols = {"Team", "G", "MP"}

# dictionary to store DataFrames by table ID
team_tables = {}

In [ ]:
def load_table(table_id, soup, comments):
    # try to find visible table
    tag = soup.find("table", {"id": table_id})

    # if not visible, look in comments
    if tag is None:
        for c in comments:
            if f'id="{table_id}"' in c:
                tag = BeautifulSoup(c, "html.parser").find("table", {"id": table_id})
                break
    if tag is None:
        print(f"Table not found: {table_id}")
        return None

    # shooting tables use multilevel headers
    if "shooting" in table_id:
        df = pd.read_html(StringIO(str(tag)), header=[0, 1])[0]
        df.columns = [f"{a}_{b}" if not a.startswith("Unnamed") else b for a, b in df.columns]
    else:
        df = pd.read_html(StringIO(str(tag)), header=0)[0]

    # prefix columns except base ones
    prefix = table_id.split("-")[0]
    df.columns = [col if col in no_prefix_cols else f"{prefix}_{col}" for col in df.columns]
    return df

In [ ]:
team_tables = []

for table_id in table_ids:
    print(f"Loading: {table_id}")
    df = load_table(table_id, soup, comments)
    if df is not None:
        team_tables.append((table_id, df))

Loading: per_game-team
Loading: per_game-opponent
Loading: totals-team
Loading: totals-opponent
Loading: advanced-team
Loading: per_poss-team
Loading: per_poss-opponent
Loading: shooting-team
Loading: shooting-opponent


In [ ]:
for name, df in team_tables:
    print(f"{name}: {df.shape}")

per_game-team: (13, 25)
per_game-opponent: (13, 25)
totals-team: (13, 25)
totals-opponent: (13, 25)
advanced-team: (14, 29)
per_poss-team: (12, 25)
per_poss-opponent: (12, 25)
shooting-team: (13, 26)
shooting-opponent: (13, 26)


In [ ]:
for name, df in team_tables:
    if name == "per_game-team":
        print(name, df.shape)
        display(df.head())
        break

per_game-team (13, 24)


,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,...,per_game_FT%,per_game_ORB,per_game_DRB,per_game_TRB,per_game_AST,per_game_STL,per_game_BLK,per_game_TOV,per_game_PF,per_game_PTS
0,Las Vegas Aces*,40,200.6,30.9,68.1,0.454,9.4,26.5,0.355,21.5,...,0.828,5.6,28.5,34.1,20.5,7.1,5.0,10.8,16.5,86.4
1,New York Liberty*,40,200.0,30.8,68.7,0.448,10.1,29.0,0.349,20.6,...,0.814,8.5,28.1,36.6,22.8,7.9,4.5,12.7,15.4,85.6
2,Indiana Fever*,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.775,8.3,26.8,35.1,20.4,5.9,4.3,14.2,18.2,85.0
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.786,10.5,24.3,34.8,20.4,7.1,4.0,14.8,18.5,84.2
4,Seattle Storm*,40,201.2,31.1,71.3,0.435,6.1,21.0,0.288,25.0,...,0.840,8.7,26.0,34.7,20.7,9.3,5.2,12.4,16.5,83.2


In [ ]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["per_game_Rk"]) if name == "per_game-team" and "per_game_Rk" in df.columns else df)
               for name, df in team_tables]

In [ ]:
for name, df in team_tables:
    if name == "per_game-team":
        print(name, df.shape)
        display(df.head())
        break

per_game-team (13, 24)


,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,...,per_game_FT%,per_game_ORB,per_game_DRB,per_game_TRB,per_game_AST,per_game_STL,per_game_BLK,per_game_TOV,per_game_PF,per_game_PTS
0,Las Vegas Aces*,40,200.6,30.9,68.1,0.454,9.4,26.5,0.355,21.5,...,0.828,5.6,28.5,34.1,20.5,7.1,5.0,10.8,16.5,86.4
1,New York Liberty*,40,200.0,30.8,68.7,0.448,10.1,29.0,0.349,20.6,...,0.814,8.5,28.1,36.6,22.8,7.9,4.5,12.7,15.4,85.6
2,Indiana Fever*,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.775,8.3,26.8,35.1,20.4,5.9,4.3,14.2,18.2,85.0
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.786,10.5,24.3,34.8,20.4,7.1,4.0,14.8,18.5,84.2
4,Seattle Storm*,40,201.2,31.1,71.3,0.435,6.1,21.0,0.288,25.0,...,0.840,8.7,26.0,34.7,20.7,9.3,5.2,12.4,16.5,83.2


In [ ]:
for name, df in team_tables:
    if name == "per_game-opponent":
        print(name, df.shape)
        display(df.head())
        break

per_game-opponent (13, 25)


,per_game_Rk,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,...,per_game_FT%,per_game_ORB,per_game_DRB,per_game_TRB,per_game_AST,per_game_STL,per_game_BLK,per_game_TOV,per_game_PF,per_game_PTS
0,1.0,Connecticut Sun*,40,201.2,27.2,63.0,0.431,6.5,20.6,0.313,...,0.779,7.0,24.7,31.7,19.1,7.0,4.6,15.0,18.1,73.6
1,2.0,Minnesota Lynx*,40,201.9,28.2,68.7,0.410,6.8,22.7,0.301,...,0.788,9.4,25.9,35.3,18.6,7.5,3.7,14.9,15.7,75.6
2,3.0,New York Liberty*,40,200.0,29.1,68.3,0.425,7.0,21.5,0.324,...,0.769,7.4,25.3,32.7,19.4,6.2,3.2,12.7,16.9,76.5
3,4.0,Seattle Storm*,40,201.2,28.8,67.7,0.426,6.9,21.0,0.329,...,0.784,8.9,27.2,36.0,19.5,7.6,4.6,15.0,16.4,78.8
4,5.0,Atlanta Dream*,40,201.9,28.9,67.4,0.429,8.0,23.1,0.344,...,0.797,7.5,27.2,34.6,20.1,6.9,4.1,12.5,17.6,79.8


In [ ]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["per_game_Rk"]) if name == "per_game-opponent" and "per_game_Rk" in df.columns else df)
               for name, df in team_tables]

In [ ]:
for name, df in team_tables:
    if name == "per_game-opponent":
        print(name, df.shape)
        display(df.head())
        break

per_game-opponent (13, 24)


,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,...,per_game_FT%,per_game_ORB,per_game_DRB,per_game_TRB,per_game_AST,per_game_STL,per_game_BLK,per_game_TOV,per_game_PF,per_game_PTS
0,Connecticut Sun*,40,201.2,27.2,63.0,0.431,6.5,20.6,0.313,20.7,...,0.779,7.0,24.7,31.7,19.1,7.0,4.6,15.0,18.1,73.6
1,Minnesota Lynx*,40,201.9,28.2,68.7,0.410,6.8,22.7,0.301,21.4,...,0.788,9.4,25.9,35.3,18.6,7.5,3.7,14.9,15.7,75.6
2,New York Liberty*,40,200.0,29.1,68.3,0.425,7.0,21.5,0.324,22.1,...,0.769,7.4,25.3,32.7,19.4,6.2,3.2,12.7,16.9,76.5
3,Seattle Storm*,40,201.2,28.8,67.7,0.426,6.9,21.0,0.329,21.9,...,0.784,8.9,27.2,36.0,19.5,7.6,4.6,15.0,16.4,78.8
4,Atlanta Dream*,40,201.9,28.9,67.4,0.429,8.0,23.1,0.344,21.0,...,0.797,7.5,27.2,34.6,20.1,6.9,4.1,12.5,17.6,79.8


In [ ]:
for name, df in team_tables:
    if name == "totals-team":
        print(name, df.shape)
        display(df.head())
        break

totals-team (13, 25)


,totals_Rk,Team,G,MP,totals_FG,totals_FGA,totals_FG%,totals_3P,totals_3PA,totals_3P%,...,totals_FT%,totals_ORB,totals_DRB,totals_TRB,totals_AST,totals_STL,totals_BLK,totals_TOV,totals_PF,totals_PTS
0,1.0,Las Vegas Aces*,40,8024,1236,2725,0.454,376,1058,0.355,...,0.828,223,1141,1364,820,282,198,432,661,3455
1,2.0,New York Liberty*,40,7999,1230,2747,0.448,405,1160,0.349,...,0.814,338,1125,1463,911,316,179,506,615,3424
2,3.0,Indiana Fever*,40,8024,1250,2741,0.456,368,1034,0.356,...,0.775,331,1071,1402,816,235,173,568,726,3399
3,4.0,Dallas Wings,40,8074,1266,2841,0.446,250,767,0.326,...,0.786,419,971,1390,817,285,158,593,739,3368
4,5.0,Seattle Storm*,40,8049,1242,2852,0.435,242,840,0.288,...,0.840,346,1040,1386,826,372,206,496,660,3329


In [ ]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["totals_Rk"]) if name == "totals-team" and "totals_Rk" in df.columns else df)
               for name, df in team_tables]

In [ ]:
for name, df in team_tables:
    if name == "totals-team":
        print(name, df.shape)
        display(df.head())
        break

totals-team (13, 24)


,Team,G,MP,totals_FG,totals_FGA,totals_FG%,totals_3P,totals_3PA,totals_3P%,totals_2P,...,totals_FT%,totals_ORB,totals_DRB,totals_TRB,totals_AST,totals_STL,totals_BLK,totals_TOV,totals_PF,totals_PTS
0,Las Vegas Aces*,40,8024,1236,2725,0.454,376,1058,0.355,860,...,0.828,223,1141,1364,820,282,198,432,661,3455
1,New York Liberty*,40,7999,1230,2747,0.448,405,1160,0.349,825,...,0.814,338,1125,1463,911,316,179,506,615,3424
2,Indiana Fever*,40,8024,1250,2741,0.456,368,1034,0.356,882,...,0.775,331,1071,1402,816,235,173,568,726,3399
3,Dallas Wings,40,8074,1266,2841,0.446,250,767,0.326,1016,...,0.786,419,971,1390,817,285,158,593,739,3368
4,Seattle Storm*,40,8049,1242,2852,0.435,242,840,0.288,1000,...,0.840,346,1040,1386,826,372,206,496,660,3329


In [ ]:
for name, df in team_tables:
    if name == "totals-opponent":
        print(name, df.shape)
        display(df.head())
        break

totals-opponent (13, 25)


,totals_Rk,Team,G,MP,totals_FG,totals_FGA,totals_FG%,totals_3P,totals_3PA,totals_3P%,...,totals_FT%,totals_ORB,totals_DRB,totals_TRB,totals_AST,totals_STL,totals_BLK,totals_TOV,totals_PF,totals_PTS
0,1.0,Connecticut Sun*,40,8049,1086.0,2520.0,0.431,258.0,825.0,0.313,...,0.779,280.0,989.0,1269.0,764.0,280.0,183.0,599.0,725.0,2944.0
1,2.0,Minnesota Lynx*,40,8074,1127.0,2749.0,0.410,273.0,907.0,0.301,...,0.788,374.0,1037.0,1411.0,742.0,298.0,146.0,596.0,626.0,3024.0
2,3.0,New York Liberty*,40,7999,1162.0,2732.0,0.425,279.0,860.0,0.324,...,0.769,296.0,1010.0,1306.0,777.0,249.0,127.0,506.0,674.0,3058.0
3,4.0,Seattle Storm*,40,8049,1152.0,2707.0,0.426,277.0,841.0,0.329,...,0.784,355.0,1086.0,1441.0,779.0,305.0,183.0,599.0,656.0,3150.0
4,5.0,Atlanta Dream*,40,8074,1156.0,2695.0,0.429,318.0,924.0,0.344,...,0.797,298.0,1086.0,1384.0,804.0,275.0,164.0,501.0,702.0,3190.0


In [ ]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["totals_Rk"]) if name == "totals-opponent" and "totals_Rk" in df.columns else df)
               for name, df in team_tables]

In [ ]:
for name, df in team_tables:
    if name == "totals-opponent":
        print(name, df.shape)
        display(df.head())
        break

totals-opponent (13, 24)


,Team,G,MP,totals_FG,totals_FGA,totals_FG%,totals_3P,totals_3PA,totals_3P%,totals_2P,...,totals_FT%,totals_ORB,totals_DRB,totals_TRB,totals_AST,totals_STL,totals_BLK,totals_TOV,totals_PF,totals_PTS
0,Connecticut Sun*,40,8049,1086.0,2520.0,0.431,258.0,825.0,0.313,828.0,...,0.779,280.0,989.0,1269.0,764.0,280.0,183.0,599.0,725.0,2944.0
1,Minnesota Lynx*,40,8074,1127.0,2749.0,0.410,273.0,907.0,0.301,854.0,...,0.788,374.0,1037.0,1411.0,742.0,298.0,146.0,596.0,626.0,3024.0
2,New York Liberty*,40,7999,1162.0,2732.0,0.425,279.0,860.0,0.324,883.0,...,0.769,296.0,1010.0,1306.0,777.0,249.0,127.0,506.0,674.0,3058.0
3,Seattle Storm*,40,8049,1152.0,2707.0,0.426,277.0,841.0,0.329,875.0,...,0.784,355.0,1086.0,1441.0,779.0,305.0,183.0,599.0,656.0,3150.0
4,Atlanta Dream*,40,8074,1156.0,2695.0,0.429,318.0,924.0,0.344,838.0,...,0.797,298.0,1086.0,1384.0,804.0,275.0,164.0,501.0,702.0,3190.0


In [ ]:
for name, df in team_tables:
    if name == "per_poss-team":
        print(name, df.shape)
        display(df.head())
        break

per_poss-team (12, 25)


,per_poss_Rk,Team,G,MP,per_poss_FG,per_poss_FGA,per_poss_FG%,per_poss_3P,per_poss_3PA,per_poss_3P%,...,per_poss_FT%,per_poss_ORB,per_poss_DRB,per_poss_TRB,per_poss_AST,per_poss_STL,per_poss_BLK,per_poss_TOV,per_poss_PF,per_poss_PTS
0,1,New York Liberty*,40,7999,39.4,88.0,0.448,13.0,37.1,0.349,...,0.814,10.8,36.0,46.8,29.2,10.1,5.7,16.2,19.7,109.6
1,2,Las Vegas Aces*,40,8024,38.6,85.2,0.454,11.8,33.1,0.355,...,0.828,7.0,35.7,42.6,25.6,8.8,6.2,13.5,20.7,108.0
2,3,Indiana Fever*,40,8024,39.0,85.6,0.456,11.5,32.3,0.356,...,0.775,10.3,33.4,43.8,25.5,7.3,5.4,17.7,22.7,106.1
3,4,Connecticut Sun*,40,8049,38.3,86.4,0.444,7.7,23.6,0.327,...,0.753,10.9,32.9,43.8,26.1,10.7,4.8,15.8,21.1,105.0
4,5,Minnesota Lynx*,40,8074,38.4,85.9,0.448,12.1,31.9,0.380,...,0.790,9.5,34.2,43.7,29.4,10.9,5.4,17.1,20.9,104.6


In [ ]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["per_poss_Rk"]) if name == "per_poss-team" and "per_poss_Rk" in df.columns else df)
               for name, df in team_tables]

In [ ]:
for name, df in team_tables:
    if name == "per_poss-team":
        print(name, df.shape)
        display(df.head())
        break

per_poss-team (12, 24)


,Team,G,MP,per_poss_FG,per_poss_FGA,per_poss_FG%,per_poss_3P,per_poss_3PA,per_poss_3P%,per_poss_2P,...,per_poss_FT%,per_poss_ORB,per_poss_DRB,per_poss_TRB,per_poss_AST,per_poss_STL,per_poss_BLK,per_poss_TOV,per_poss_PF,per_poss_PTS
0,New York Liberty*,40,7999,39.4,88.0,0.448,13.0,37.1,0.349,26.4,...,0.814,10.8,36.0,46.8,29.2,10.1,5.7,16.2,19.7,109.6
1,Las Vegas Aces*,40,8024,38.6,85.2,0.454,11.8,33.1,0.355,26.9,...,0.828,7.0,35.7,42.6,25.6,8.8,6.2,13.5,20.7,108.0
2,Indiana Fever*,40,8024,39.0,85.6,0.456,11.5,32.3,0.356,27.5,...,0.775,10.3,33.4,43.8,25.5,7.3,5.4,17.7,22.7,106.1
3,Connecticut Sun*,40,8049,38.3,86.4,0.444,7.7,23.6,0.327,30.6,...,0.753,10.9,32.9,43.8,26.1,10.7,4.8,15.8,21.1,105.0
4,Minnesota Lynx*,40,8074,38.4,85.9,0.448,12.1,31.9,0.380,26.3,...,0.790,9.5,34.2,43.7,29.4,10.9,5.4,17.1,20.9,104.6


In [ ]:
for name, df in team_tables:
    if name == "per_poss-opponent":
        print(name, df.shape)
        display(df.head())
        break

per_poss-opponent (12, 25)


,per_poss_Rk,Team,G,MP,per_poss_FG,per_poss_FGA,per_poss_FG%,per_poss_3P,per_poss_3PA,per_poss_3P%,...,per_poss_FT%,per_poss_ORB,per_poss_DRB,per_poss_TRB,per_poss_AST,per_poss_STL,per_poss_BLK,per_poss_TOV,per_poss_PF,per_poss_PTS
0,1,Connecticut Sun*,40,8049,35.6,82.6,0.431,8.5,27.0,0.313,...,0.779,9.2,32.4,41.6,25.0,9.2,6.0,19.6,23.8,96.4
1,2,Minnesota Lynx*,40,8074,36.0,87.7,0.410,8.7,28.9,0.301,...,0.788,11.9,33.1,45.0,23.7,9.5,4.7,19.0,20.0,96.5
2,3,New York Liberty*,40,7999,37.2,87.5,0.425,8.9,27.5,0.324,...,0.769,9.5,32.3,41.8,24.9,8.0,4.1,16.2,21.6,97.9
3,4,Seattle Storm*,40,8049,36.0,84.7,0.426,8.7,26.3,0.329,...,0.784,11.1,34.0,45.1,24.4,9.5,5.7,18.7,20.5,98.6
4,5,Las Vegas Aces*,40,8024,37.6,86.9,0.433,9.6,27.5,0.350,...,0.761,9.2,35.1,44.3,24.9,8.1,4.5,15.8,21.5,101.2


In [ ]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["per_poss_Rk"]) if name == "per_poss-opponent" and "per_poss_Rk" in df.columns else df)
               for name, df in team_tables]

In [ ]:
for name, df in team_tables:
    if name == "per_poss-opponent":
        print(name, df.shape)
        display(df.head())
        break

per_poss-opponent (12, 24)


,Team,G,MP,per_poss_FG,per_poss_FGA,per_poss_FG%,per_poss_3P,per_poss_3PA,per_poss_3P%,per_poss_2P,...,per_poss_FT%,per_poss_ORB,per_poss_DRB,per_poss_TRB,per_poss_AST,per_poss_STL,per_poss_BLK,per_poss_TOV,per_poss_PF,per_poss_PTS
0,Connecticut Sun*,40,8049,35.6,82.6,0.431,8.5,27.0,0.313,27.1,...,0.779,9.2,32.4,41.6,25.0,9.2,6.0,19.6,23.8,96.4
1,Minnesota Lynx*,40,8074,36.0,87.7,0.410,8.7,28.9,0.301,27.2,...,0.788,11.9,33.1,45.0,23.7,9.5,4.7,19.0,20.0,96.5
2,New York Liberty*,40,7999,37.2,87.5,0.425,8.9,27.5,0.324,28.3,...,0.769,9.5,32.3,41.8,24.9,8.0,4.1,16.2,21.6,97.9
3,Seattle Storm*,40,8049,36.0,84.7,0.426,8.7,26.3,0.329,27.4,...,0.784,11.1,34.0,45.1,24.4,9.5,5.7,18.7,20.5,98.6
4,Las Vegas Aces*,40,8024,37.6,86.9,0.433,9.6,27.5,0.350,27.9,...,0.761,9.2,35.1,44.3,24.9,8.1,4.5,15.8,21.5,101.2


In [ ]:
for name, df in team_tables:
    if name == "shooting-team":
        print(name, df.shape)
        display(df.head())
        break

shooting-team (13, 26)


,shooting_Rk,Team,G,MP,shooting_FG%,shooting_Dist.,shooting_Unnamed: 6_level_1,shooting_% of FGA by Distance_2P,shooting_% of FGA by Distance_0-3,shooting_% of FGA by Distance_3-10,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,1.0,Atlanta Dream*,40,8074,0.408,12.8,NaN,0.715,0.228,0.226,...,0.374,0.339,0.395,0.308,NaN,0.599,0.891,NaN,0.210,0.288
1,2.0,Chicago Sky,40,8000,0.422,10.8,NaN,0.788,0.298,0.262,...,0.377,0.358,0.344,0.324,NaN,0.606,0.808,NaN,0.195,0.310
2,3.0,Connecticut Sun*,40,8049,0.444,11.9,NaN,0.727,0.267,0.262,...,0.416,0.414,0.345,0.327,NaN,0.635,0.860,NaN,0.181,0.300
3,4.0,Dallas Wings,40,8074,0.446,12.3,NaN,0.730,0.242,0.250,...,0.443,0.337,0.383,0.326,NaN,0.614,0.772,NaN,0.149,0.333
4,5.0,Indiana Fever*,40,8024,0.456,13.6,NaN,0.623,0.280,0.189,...,0.478,0.375,0.372,0.356,NaN,0.602,0.774,NaN,0.182,0.372


In [ ]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["shooting_Rk"]) if name == "shooting-team" and "shooting_Rk" in df.columns else df)
               for name, df in team_tables]

In [ ]:
for name, df in team_tables:
    if name == "shooting-team":
        print(name, df.shape)
        display(df.head())
        break

shooting-team (13, 25)


,Team,G,MP,shooting_FG%,shooting_Dist.,shooting_Unnamed: 6_level_1,shooting_% of FGA by Distance_2P,shooting_% of FGA by Distance_0-3,shooting_% of FGA by Distance_3-10,shooting_% of FGA by Distance_10-16,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,Atlanta Dream*,40,8074,0.408,12.8,NaN,0.715,0.228,0.226,0.137,...,0.374,0.339,0.395,0.308,NaN,0.599,0.891,NaN,0.210,0.288
1,Chicago Sky,40,8000,0.422,10.8,NaN,0.788,0.298,0.262,0.120,...,0.377,0.358,0.344,0.324,NaN,0.606,0.808,NaN,0.195,0.310
2,Connecticut Sun*,40,8049,0.444,11.9,NaN,0.727,0.267,0.262,0.110,...,0.416,0.414,0.345,0.327,NaN,0.635,0.860,NaN,0.181,0.300
3,Dallas Wings,40,8074,0.446,12.3,NaN,0.730,0.242,0.250,0.130,...,0.443,0.337,0.383,0.326,NaN,0.614,0.772,NaN,0.149,0.333
4,Indiana Fever*,40,8024,0.456,13.6,NaN,0.623,0.280,0.189,0.073,...,0.478,0.375,0.372,0.356,NaN,0.602,0.774,NaN,0.182,0.372


In [ ]:
for name, df in team_tables:
    if name == "shooting-opponent":
        print(name, df.shape)
        display(df.head())
        break

shooting-opponent (13, 26)


,shooting_Rk,Team,G,MP,shooting_FG%,shooting_Dist.,shooting_Unnamed: 6_level_1,shooting_% of FGA by Distance_2P,shooting_% of FGA by Distance_0-3,shooting_% of FGA by Distance_3-10,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,1.0,Atlanta Dream*,40,8075,0.429,13.0,NaN,0.657,0.243,0.223,...,0.400,0.358,0.397,0.344,NaN,0.628,0.874,NaN,0.215,0.357
1,2.0,Chicago Sky,40,8000,0.446,13.0,NaN,0.678,0.275,0.200,...,0.391,0.393,0.433,0.326,NaN,0.672,0.880,NaN,0.203,0.278
2,3.0,Connecticut Sun*,40,8050,0.431,13.0,NaN,0.673,0.207,0.257,...,0.435,0.401,0.362,0.313,NaN,0.644,0.895,NaN,0.193,0.346
3,4.0,Dallas Wings,40,8075,0.475,13.0,NaN,0.668,0.232,0.236,...,0.468,0.400,0.397,0.365,NaN,0.620,0.898,NaN,0.197,0.357
4,5.0,Indiana Fever*,40,8025,0.441,14.0,NaN,0.638,0.228,0.209,...,0.448,0.375,0.408,0.361,NaN,0.603,0.889,NaN,0.193,0.396


In [ ]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["shooting_Rk"]) if name == "shooting-opponent" and "shooting_Rk" in df.columns else df)
               for name, df in team_tables]

In [ ]:
for name, df in team_tables:
    if name == "shooting-opponent":
        print(name, df.shape)
        display(df.head())
        break

shooting-opponent (13, 25)


,Team,G,MP,shooting_FG%,shooting_Dist.,shooting_Unnamed: 6_level_1,shooting_% of FGA by Distance_2P,shooting_% of FGA by Distance_0-3,shooting_% of FGA by Distance_3-10,shooting_% of FGA by Distance_10-16,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,Atlanta Dream*,40,8075,0.429,13.0,NaN,0.657,0.243,0.223,0.102,...,0.400,0.358,0.397,0.344,NaN,0.628,0.874,NaN,0.215,0.357
1,Chicago Sky,40,8000,0.446,13.0,NaN,0.678,0.275,0.200,0.100,...,0.391,0.393,0.433,0.326,NaN,0.672,0.880,NaN,0.203,0.278
2,Connecticut Sun*,40,8050,0.431,13.0,NaN,0.673,0.207,0.257,0.120,...,0.435,0.401,0.362,0.313,NaN,0.644,0.895,NaN,0.193,0.346
3,Dallas Wings,40,8075,0.475,13.0,NaN,0.668,0.232,0.236,0.106,...,0.468,0.400,0.397,0.365,NaN,0.620,0.898,NaN,0.197,0.357
4,Indiana Fever*,40,8025,0.441,14.0,NaN,0.638,0.228,0.209,0.112,...,0.448,0.375,0.408,0.361,NaN,0.603,0.889,NaN,0.193,0.396


In [ ]:
for name, df in team_tables:
    if name == "advanced-team":
        print(name, df.shape)
        display(df.head())
        break

advanced-team (14, 29)


,advanced_Unnamed: 0,advanced_Unnamed: 1,advanced_Unnamed: 2,advanced_Unnamed: 3,advanced_Unnamed: 4,advanced_Unnamed: 5,advanced_Unnamed: 6,advanced_Unnamed: 7,advanced_Unnamed: 8,advanced_Unnamed: 9,...,advanced_Offense Four Factors.1,advanced_Offense Four Factors.2,advanced_Offense Four Factors.3,advanced_Unnamed: 22,advanced_Defense Four Factors,advanced_Defense Four Factors.1,advanced_Defense Four Factors.2,advanced_Defense Four Factors.3,advanced_Unnamed: 27,advanced_Unnamed: 28
0,Rk,Team,Age,W,L,PW,PL,MOV,SOS,SRS,...,TOV%,ORB%,FT/FGA,NaN,eFG%,TOV%,DRB%,FT/FGA,NaN,Arena
1,1,New York Liberty*,28.5,32,8,33,7,9.15,-1.09,8.06,...,14.2,25.1,.203,NaN,.476,14.5,79.2,.167,NaN,NaN
2,2,Connecticut Sun*,28.9,28,12,31,9,6.50,-0.75,5.75,...,13.8,25.2,.239,NaN,.482,17.6,78.2,.204,NaN,NaN
3,3,Minnesota Lynx*,27.9,30,10,30,10,6.38,-0.74,5.64,...,15.3,22.3,.182,NaN,.460,16.5,74.2,.181,NaN,NaN
4,4,Las Vegas Aces*,29.6,27,13,29,11,5.48,-0.70,4.77,...,12.4,16.6,.223,NaN,.488,14.1,79.5,.189,NaN,NaN


In [ ]:
# flatten advanced-team header and update in place
for i, (name, df) in enumerate(team_tables):
    if name == "advanced-team":
        df.columns = df.iloc[0]  # use row 0 as header
        df = df[1:].reset_index(drop=True)  # drop that header row
        team_tables[i] = (name, df)  # update in list

In [ ]:
for name, df in team_tables:
    if name == "advanced-team":
        print(name, df.shape)
        display(df.head())
        break

advanced-team (13, 29)


,Rk,Team,Age,W,L,PW,PL,MOV,SOS,SRS,...,TOV%,ORB%,FT/FGA,NaN,eFG%,TOV%,DRB%,FT/FGA,NaN,Arena
0,1,New York Liberty*,28.5,32,8,33,7,9.15,-1.09,8.06,...,14.2,25.1,.203,NaN,.476,14.5,79.2,.167,NaN,NaN
1,2,Connecticut Sun*,28.9,28,12,31,9,6.50,-0.75,5.75,...,13.8,25.2,.239,NaN,.482,17.6,78.2,.204,NaN,NaN
2,3,Minnesota Lynx*,27.9,30,10,30,10,6.38,-0.74,5.64,...,15.3,22.3,.182,NaN,.460,16.5,74.2,.181,NaN,NaN
3,4,Las Vegas Aces*,29.6,27,13,29,11,5.48,-0.70,4.77,...,12.4,16.6,.223,NaN,.488,14.1,79.5,.189,NaN,NaN
4,5,Seattle Storm*,29.1,25,15,27,13,4.48,-0.56,3.92,...,13.5,24.2,.211,NaN,.477,16.5,74.6,.210,NaN,NaN


In [ ]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["Rk"]) if name == "advanced-team" and "Rk" in df.columns else df)
               for name, df in team_tables]

In [ ]:
for name, df in team_tables:
    if name == "advanced-team":
        print(name, df.shape)
        display(df.head())
        break

advanced-team (13, 28)


,Team,Age,W,L,PW,PL,MOV,SOS,SRS,ORtg,...,TOV%,ORB%,FT/FGA,NaN,eFG%,TOV%,DRB%,FT/FGA,NaN,Arena
0,New York Liberty*,28.5,32,8,33,7,9.15,-1.09,8.06,109.6,...,14.2,25.1,.203,NaN,.476,14.5,79.2,.167,NaN,NaN
1,Connecticut Sun*,28.9,28,12,31,9,6.50,-0.75,5.75,105.0,...,13.8,25.2,.239,NaN,.482,17.6,78.2,.204,NaN,NaN
2,Minnesota Lynx*,27.9,30,10,30,10,6.38,-0.74,5.64,104.6,...,15.3,22.3,.182,NaN,.460,16.5,74.2,.181,NaN,NaN
3,Las Vegas Aces*,29.6,27,13,29,11,5.48,-0.70,4.77,108.0,...,12.4,16.6,.223,NaN,.488,14.1,79.5,.189,NaN,NaN
4,Seattle Storm*,29.1,25,15,27,13,4.48,-0.56,3.92,104.2,...,13.5,24.2,.211,NaN,.477,16.5,74.6,.210,NaN,NaN


In [ ]:
# remove trailing '*' from all Team names
for i, (name, df) in enumerate(team_tables):
    if "Team" in df.columns:
        df["Team"] = df["Team"].str.replace("*", "", regex=False).str.strip()
        team_tables[i] = (name, df)

advanced-team (13, 28)


,Team,Age,W,L,PW,PL,MOV,SOS,SRS,ORtg,...,TOV%,ORB%,FT/FGA,NaN,eFG%,TOV%,DRB%,FT/FGA,NaN,Arena
0,New York Liberty,28.5,32,8,33,7,9.15,-1.09,8.06,109.6,...,14.2,25.1,.203,NaN,.476,14.5,79.2,.167,NaN,NaN
1,Connecticut Sun,28.9,28,12,31,9,6.50,-0.75,5.75,105.0,...,13.8,25.2,.239,NaN,.482,17.6,78.2,.204,NaN,NaN
2,Minnesota Lynx,27.9,30,10,30,10,6.38,-0.74,5.64,104.6,...,15.3,22.3,.182,NaN,.460,16.5,74.2,.181,NaN,NaN
3,Las Vegas Aces,29.6,27,13,29,11,5.48,-0.70,4.77,108.0,...,12.4,16.6,.223,NaN,.488,14.1,79.5,.189,NaN,NaN
4,Seattle Storm,29.1,25,15,27,13,4.48,-0.56,3.92,104.2,...,13.5,24.2,.211,NaN,.477,16.5,74.6,.210,NaN,NaN


In [ ]:
# columns to drop from all tables except per_game-team
shared_cols = ["G", "MP"]

# drop G and MP from every table except per_game-team
for name, df in team_tables:
    if name != "per_game-team":
        df.drop(columns=[col for col in shared_cols if col in df.columns], inplace=True)

In [ ]:
# separate into team and opponent tables
team_frames = [df for name, df in team_tables if name.endswith("-team")]
opp_frames = [df for name, df in team_tables if name.endswith("-opponent")]

# merge them
df_team = reduce(lambda left, right: pd.merge(left, right, on="Team", how="outer"), team_frames)
df_opp = reduce(lambda left, right: pd.merge(left, right, on="Team", how="outer"), opp_frames)

In [ ]:
df_team.head()

,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,Atlanta Dream,40,201.9,27.8,68.1,0.408,6.0,19.4,0.308,21.8,...,0.374,0.339,0.395,0.308,NaN,0.599,0.891,NaN,0.210,0.288
1,Chicago Sky,40,200.0,29.7,70.3,0.422,4.8,14.9,0.323,24.9,...,0.377,0.358,0.344,0.324,NaN,0.606,0.808,NaN,0.195,0.310
2,Connecticut Sun,40,201.2,29.3,65.9,0.444,5.9,18.0,0.327,23.4,...,0.416,0.414,0.345,0.327,NaN,0.635,0.860,NaN,0.181,0.300
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.443,0.337,0.383,0.326,NaN,0.614,0.772,NaN,0.149,0.333
4,Indiana Fever,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.478,0.375,0.372,0.356,NaN,0.602,0.774,NaN,0.182,0.372


In [ ]:
df.shape

(13, 23)

In [ ]:
# show null counts for each column in df_team
df_team.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0
NaN,13
Arena,13
NaN,13
NaN,13
shooting_Unnamed: 23_level_1,13
shooting_Unnamed: 20_level_1,13
shooting_Unnamed: 13_level_1,13
shooting_Unnamed: 6_level_1,13
per_poss_FGA,1
per_poss_FG%,1


In [ ]:
# drop unwanted columns from df_team
df_team.drop(columns=[
    "Arena",
    "NaN",
    "shooting_Unnamed: 23_level_1",
    "shooting_Unnamed: 20_level_1",
    "shooting_Unnamed: 13_level_1",
    "shooting_Unnamed: 6_level_1"
], inplace=True)

KeyError: "['Arena' 'NaN' 'shooting_Unnamed: 23_level_1'\n 'shooting_Unnamed: 20_level_1' 'shooting_Unnamed: 13_level_1'\n 'shooting_Unnamed: 6_level_1'] not found in axis"

In [ ]:
# drop columns with name NaN
df_team.drop(columns=[col for col in df_team.columns if pd.isna(col)], inplace=True)

In [ ]:
for col in df_team.columns:
    print(repr(col))

'Team'
'G'
'MP'
'per_game_FG'
'per_game_FGA'
'per_game_FG%'
'per_game_3P'
'per_game_3PA'
'per_game_3P%'
'per_game_2P'
'per_game_2PA'
'per_game_2P%'
'per_game_FT'
'per_game_FTA'
'per_game_FT%'
'per_game_ORB'
'per_game_DRB'
'per_game_TRB'
'per_game_AST'
'per_game_STL'
'per_game_BLK'
'per_game_TOV'
'per_game_PF'
'per_game_PTS'
'totals_FG'
'totals_FGA'
'totals_FG%'
'totals_3P'
'totals_3PA'
'totals_3P%'
'totals_2P'
'totals_2PA'
'totals_2P%'
'totals_FT'
'totals_FTA'
'totals_FT%'
'totals_ORB'
'totals_DRB'
'totals_TRB'
'totals_AST'
'totals_STL'
'totals_BLK'
'totals_TOV'
'totals_PF'
'totals_PTS'
'Age'
'W'
'L'
'PW'
'PL'
'MOV'
'SOS'
'SRS'
'ORtg'
'DRtg'
'NRtg'
'Pace'
'FTr'
'3PAr'
'TS%'
np.float64(nan)
'eFG%'
'TOV%'
'ORB%'
'FT/FGA'
np.float64(nan)
'eFG%'
'TOV%'
'DRB%'
'FT/FGA'
np.float64(nan)
'per_poss_FG'
'per_poss_FGA'
'per_poss_FG%'
'per_poss_3P'
'per_poss_3PA'
'per_poss_3P%'
'per_poss_2P'
'per_poss_2PA'
'per_poss_2P%'
'per_poss_FT'
'per_poss_FTA'
'per_poss_FT%'
'per_poss_ORB'
'per_poss_DRB'
'pe

In [ ]:
df_team = df_team.loc[:, [not isinstance(col, float) or not np.isnan(col) for col in df_team.columns]]

In [ ]:
# show null counts for each column in df_team
df_team.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0
W,1
L,1
NRtg,1
per_poss_FG,1
per_poss_FGA,1
per_poss_FG%,1
per_poss_3P,1
per_poss_3PA,1
per_poss_3P%,1
per_poss_2P,1


In [ ]:
df_team[df_team.isnull().any(axis=1)]

,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,...,shooting_FG% by Distance_2P,shooting_FG% by Distance_0-3,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Corner_%3PA,shooting_Corner_3P%
6,League Average,40,201.0,29.9,68.3,0.438,7.7,22.8,0.338,22.2,...,0.488,0.654,0.432,0.386,0.379,0.338,0.62,0.876,0.19,0.356


In [ ]:
df_team = df_team[df_team["Team"] != "League Average"]

In [ ]:
# show null counts for each column in df_team
df_team.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0


In [ ]:
for col in df_team.columns:
    print(repr(col))

'Team'
'G'
'MP'
'per_game_FG'
'per_game_FGA'
'per_game_FG%'
'per_game_3P'
'per_game_3PA'
'per_game_3P%'
'per_game_2P'
'per_game_2PA'
'per_game_2P%'
'per_game_FT'
'per_game_FTA'
'per_game_FT%'
'per_game_ORB'
'per_game_DRB'
'per_game_TRB'
'per_game_AST'
'per_game_STL'
'per_game_BLK'
'per_game_TOV'
'per_game_PF'
'per_game_PTS'
'totals_FG'
'totals_FGA'
'totals_FG%'
'totals_3P'
'totals_3PA'
'totals_3P%'
'totals_2P'
'totals_2PA'
'totals_2P%'
'totals_FT'
'totals_FTA'
'totals_FT%'
'totals_ORB'
'totals_DRB'
'totals_TRB'
'totals_AST'
'totals_STL'
'totals_BLK'
'totals_TOV'
'totals_PF'
'totals_PTS'
'Age'
'W'
'L'
'PW'
'PL'
'MOV'
'SOS'
'SRS'
'ORtg'
'DRtg'
'NRtg'
'Pace'
'FTr'
'3PAr'
'TS%'
'eFG%'
'TOV%'
'ORB%'
'FT/FGA'
'eFG%'
'TOV%'
'DRB%'
'FT/FGA'
'per_poss_FG'
'per_poss_FGA'
'per_poss_FG%'
'per_poss_3P'
'per_poss_3PA'
'per_poss_3P%'
'per_poss_2P'
'per_poss_2PA'
'per_poss_2P%'
'per_poss_FT'
'per_poss_FTA'
'per_poss_FT%'
'per_poss_ORB'
'per_poss_DRB'
'per_poss_TRB'
'per_poss_AST'
'per_poss_STL'
'per_p

In [ ]:
df_opp.head()

,Team,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,per_game_2PA,per_game_2P%,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,Atlanta Dream,28.9,67.4,0.429,8.0,23.1,0.344,21.0,44.3,0.473,...,0.400,0.358,0.397,0.344,NaN,0.628,0.874,NaN,0.215,0.357
1,Chicago Sky,30.1,67.4,0.446,7.1,21.7,0.326,23.0,45.7,0.503,...,0.391,0.393,0.433,0.326,NaN,0.672,0.880,NaN,0.203,0.278
2,Connecticut Sun,27.2,63.0,0.431,6.5,20.6,0.313,20.7,42.4,0.488,...,0.435,0.401,0.362,0.313,NaN,0.644,0.895,NaN,0.193,0.346
3,Dallas Wings,33.5,70.5,0.475,8.6,23.4,0.365,24.9,47.1,0.530,...,0.468,0.400,0.397,0.365,NaN,0.620,0.898,NaN,0.197,0.357
4,Indiana Fever,31.2,70.6,0.441,9.2,25.5,0.361,21.9,45.1,0.487,...,0.448,0.375,0.408,0.361,NaN,0.603,0.889,NaN,0.193,0.396


In [ ]:
df_opp.shape

(13, 86)

In [ ]:
# show null counts for each column in df_team
df_opp.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0
shooting_Unnamed: 20_level_1,13
shooting_Unnamed: 13_level_1,13
shooting_Unnamed: 6_level_1,13
shooting_Unnamed: 23_level_1,13
per_game_FG%,1
...,...
per_poss_DRB,1
per_poss_PTS,1
per_poss_PF,1
per_poss_TOV,1


In [ ]:
# columns to drop from df_opp
junk_cols_opp = [
    "shooting_Unnamed: 20_level_1",
    "shooting_Unnamed: 13_level_1",
    "shooting_Unnamed: 6_level_1",
    "shooting_Unnamed: 23_level_1"
]

# drop them if they exist
df_opp.drop(columns=[col for col in junk_cols_opp if col in df_opp.columns], inplace=True)

In [ ]:
# show null counts for each column in df_team
df_opp.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0
per_game_FG,1
per_game_FGA,1
per_game_FG%,1
per_game_3P,1
per_game_3PA,1
...,...
per_poss_STL,1
per_poss_BLK,1
per_poss_TOV,1
per_poss_PF,1


In [ ]:
df_opp[df_opp.isnull().any(axis=1)]

,Team,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,per_game_2PA,per_game_2P%,...,shooting_FG% by Distance_2P,shooting_FG% by Distance_0-3,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Corner_%3PA,shooting_Corner_3P%
6,League Average,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.488,0.654,0.432,0.386,0.379,0.338,0.62,0.876,0.19,0.356


In [ ]:
df_opp = df_opp[df_opp["Team"] != "League Average"]

In [ ]:
# show null counts for each column in df_team
df_opp.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0


In [ ]:
for col in df_opp.columns:
    print(repr(col))

'Team'
'per_game_FG'
'per_game_FGA'
'per_game_FG%'
'per_game_3P'
'per_game_3PA'
'per_game_3P%'
'per_game_2P'
'per_game_2PA'
'per_game_2P%'
'per_game_FT'
'per_game_FTA'
'per_game_FT%'
'per_game_ORB'
'per_game_DRB'
'per_game_TRB'
'per_game_AST'
'per_game_STL'
'per_game_BLK'
'per_game_TOV'
'per_game_PF'
'per_game_PTS'
'totals_FG'
'totals_FGA'
'totals_FG%'
'totals_3P'
'totals_3PA'
'totals_3P%'
'totals_2P'
'totals_2PA'
'totals_2P%'
'totals_FT'
'totals_FTA'
'totals_FT%'
'totals_ORB'
'totals_DRB'
'totals_TRB'
'totals_AST'
'totals_STL'
'totals_BLK'
'totals_TOV'
'totals_PF'
'totals_PTS'
'per_poss_FG'
'per_poss_FGA'
'per_poss_FG%'
'per_poss_3P'
'per_poss_3PA'
'per_poss_3P%'
'per_poss_2P'
'per_poss_2PA'
'per_poss_2P%'
'per_poss_FT'
'per_poss_FTA'
'per_poss_FT%'
'per_poss_ORB'
'per_poss_DRB'
'per_poss_TRB'
'per_poss_AST'
'per_poss_STL'
'per_poss_BLK'
'per_poss_TOV'
'per_poss_PF'
'per_poss_PTS'
'shooting_FG%'
'shooting_Dist.'
'shooting_% of FGA by Distance_2P'
'shooting_% of FGA by Distance_0-3'
's

In [ ]:
# rename df_team columns
df_team.columns = [
    col if col in ["Team", "G", "MP"] else f"team_{col}"
    for col in df_team.columns
]

# rename df_opp columns
df_opp.columns = [
    col if col == "Team" else f"opp_{col}"
    for col in df_opp.columns
]

In [ ]:
df_team.head()

,Team,G,MP,team_per_game_FG,team_per_game_FGA,team_per_game_FG%,team_per_game_3P,team_per_game_3PA,team_per_game_3P%,team_per_game_2P,...,team_shooting_FG% by Distance_2P,team_shooting_FG% by Distance_0-3,team_shooting_FG% by Distance_3-10,team_shooting_FG% by Distance_10-16,team_shooting_FG% by Distance_16-3P,team_shooting_FG% by Distance_3P,team_shooting_% of FG Ast'd_2P,team_shooting_% of FG Ast'd_3P,team_shooting_Corner_%3PA,team_shooting_Corner_3P%
0,Atlanta Dream,40,201.9,27.8,68.1,0.408,6.0,19.4,0.308,21.8,...,0.448,0.616,0.374,0.339,0.395,0.308,0.599,0.891,0.210,0.288
1,Chicago Sky,40,200.0,29.7,70.3,0.422,4.8,14.9,0.323,24.9,...,0.449,0.587,0.377,0.358,0.344,0.324,0.606,0.808,0.195,0.310
2,Connecticut Sun,40,201.2,29.3,65.9,0.444,5.9,18.0,0.327,23.4,...,0.487,0.636,0.416,0.414,0.345,0.327,0.635,0.860,0.181,0.300
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.490,0.668,0.443,0.337,0.383,0.326,0.614,0.772,0.149,0.333
4,Indiana Fever,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.517,0.622,0.478,0.375,0.372,0.356,0.602,0.774,0.182,0.372


In [ ]:
df_opp.head()

,Team,opp_per_game_FG,opp_per_game_FGA,opp_per_game_FG%,opp_per_game_3P,opp_per_game_3PA,opp_per_game_3P%,opp_per_game_2P,opp_per_game_2PA,opp_per_game_2P%,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,Atlanta Dream,28.9,67.4,0.429,8.0,23.1,0.344,21.0,44.3,0.473,...,0.473,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,Chicago Sky,30.1,67.4,0.446,7.1,21.7,0.326,23.0,45.7,0.503,...,0.503,0.650,0.391,0.393,0.433,0.326,0.672,0.880,0.203,0.278
2,Connecticut Sun,27.2,63.0,0.431,6.5,20.6,0.313,20.7,42.4,0.488,...,0.488,0.660,0.435,0.401,0.362,0.313,0.644,0.895,0.193,0.346
3,Dallas Wings,33.5,70.5,0.475,8.6,23.4,0.365,24.9,47.1,0.530,...,0.530,0.705,0.468,0.400,0.397,0.365,0.620,0.898,0.197,0.357
4,Indiana Fever,31.2,70.6,0.441,9.2,25.5,0.361,21.9,45.1,0.487,...,0.487,0.608,0.448,0.375,0.408,0.361,0.603,0.889,0.193,0.396


In [ ]:
df = pd.merge(df_team, df_opp, on="Team", how="outer")

In [ ]:
df.head()

,Team,G,MP,team_per_game_FG,team_per_game_FGA,team_per_game_FG%,team_per_game_3P,team_per_game_3PA,team_per_game_3P%,team_per_game_2P,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,Atlanta Dream,40,201.9,27.8,68.1,0.408,6.0,19.4,0.308,21.8,...,0.473,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,Chicago Sky,40,200.0,29.7,70.3,0.422,4.8,14.9,0.323,24.9,...,0.503,0.650,0.391,0.393,0.433,0.326,0.672,0.880,0.203,0.278
2,Connecticut Sun,40,201.2,29.3,65.9,0.444,5.9,18.0,0.327,23.4,...,0.488,0.660,0.435,0.401,0.362,0.313,0.644,0.895,0.193,0.346
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.530,0.705,0.468,0.400,0.397,0.365,0.620,0.898,0.197,0.357
4,Indiana Fever,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.487,0.608,0.448,0.375,0.408,0.361,0.603,0.889,0.193,0.396


In [ ]:
df.shape

(12, 188)

In [ ]:
# save to CSV
df.to_csv("2024_team_data.csv", index=False)

# download to local machine
from google.colab import files
files.download("2024_team_data.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
def scrape_wnba_team_tables(years):
    all_data = []

    table_ids = [
        "per_game-team", "per_game-opponent",
        "totals-team", "totals-opponent",
        "advanced-team", "per_poss-team", "per_poss-opponent",
        "shooting-team", "shooting-opponent"
    ]
    no_prefix_cols = {"Team", "G", "MP"}

    for year in years:
        url = f"https://www.basketball-reference.com/wnba/years/{year}.html"
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        print(f"Processing {year}...")

        team_tables = []

        def load_table(table_id):
            tag = soup.find("table", {"id": table_id})
            if tag is None:
                for c in comments:
                    if f'id="{table_id}"' in c:
                        tag = BeautifulSoup(c, "html.parser").find("table", {"id": table_id})
                        break
            if tag is None:
                print(f"Table not found: {table_id}")
                return None

            if "shooting" in table_id:
                df = pd.read_html(StringIO(str(tag)), header=[0, 1])[0]
                df.columns = [f"{a}_{b}" if not a.startswith("Unnamed") else b for a, b in df.columns]
            else:
                df = pd.read_html(StringIO(str(tag)), header=0)[0]

            prefix = table_id.split("-")[0]
            df.columns = [col if col in no_prefix_cols else f"{prefix}_{col}" for col in df.columns]
            return df

        for table_id in table_ids:
            df = load_table(table_id)
            if df is not None:
                df = df[df["Team"] != "League Average"]
                df["Team"] = df["Team"].str.replace("*", "", regex=False).str.strip()
                team_tables.append((table_id, df))

        # Separate and merge team and opponent tables
        team_frames = [df for name, df in team_tables if name.endswith("-team")]
        opp_frames = [df for name, df in team_tables if name.endswith("-opponent")]

        # Drop shared cols from non-per_game tables
        shared_cols = ["G", "MP"]
        for name, df in team_tables:
            if name != "per_game-team":
                df.drop(columns=[col for col in shared_cols if col in df.columns and col != "Team"], inplace=True)

        from functools import reduce
        df_team = reduce(lambda left, right: pd.merge(left, right, on="Team", how="outer"), team_frames)
        df_opp = reduce(lambda left, right: pd.merge(left, right, on="Team", how="outer"), opp_frames)

        # Drop fully null columns
        df_team = df_team.loc[:, df_team.columns[df_team.notnull().any()]]
        df_opp = df_opp.loc[:, df_opp.columns[df_opp.notnull().any()]]

        # Drop League Average rows if present
        df_team = df_team[df_team["Team"] != "League Average"]
        df_opp = df_opp[df_opp["Team"] != "League Average"]

        # Rename columns
        df_team.columns = ["team_" + col if col not in ["Team", "G", "MP"] else col for col in df_team.columns]
        df_opp.columns = ["opp_" + col if col != "Team" else col for col in df_opp.columns]

        # Final merge
        df = pd.merge(df_team, df_opp, on="Team", how="outer")
        df["season"] = year
        all_data.append(df)

        time.sleep(5)

    return pd.concat(all_data, ignore_index=True)

In [5]:
years = [2020, 2021, 2022, 2023, 2024]
df_all_years = scrape_wnba_team_tables(years)

Processing 2020...


KeyError: 'Team'

In [7]:
import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment
from io import StringIO

# years to load
years = [2020, 2021, 2022, 2023]
table_ids = [
    "per_game-team", "per_game-opponent",
    "totals-team", "totals-opponent",
    "advanced-team", "per_poss-team", "per_poss-opponent",
    "shooting-team", "shooting-opponent"
]

# storage for all tables
all_tables = []

for year in years:
    print(f"Processing {year}...")
    url = f"https://www.basketball-reference.com/wnba/years/{year}.html"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.content, "html.parser")
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))

    def extract_table(table_id):
        tag = soup.find("table", {"id": table_id})
        if tag is None:
            for c in comments:
                if f'id="{table_id}"' in c:
                    tag = BeautifulSoup(c, "html.parser").find("table", {"id": table_id})
                    break
        if tag is None:
            print(f"Missing: {table_id}")
            return None

        # shooting uses multi-level header
        if "shooting" in table_id:
            df = pd.read_html(StringIO(str(tag)), header=[0, 1])[0]
            df.columns = ['_'.join(map(str, col)).strip() for col in df.columns]
        else:
            df = pd.read_html(StringIO(str(tag)), header=0)[0]
        df["table_id"] = table_id
        df["year"] = year
        return df

    for table_id in table_ids:
        df = extract_table(table_id)
        if df is not None:
            all_tables.append(df)

print(f"\nLoaded {len(all_tables)} tables.")

Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...

Loaded 36 tables.


In [9]:
# Show table_id and shape for each table
for i, df in enumerate(all_tables):
    print(f"{i}: {df['table_id'].iloc[0]} ({df['year'].iloc[0]}): {df.shape}")

0: per_game-team (2020): (13, 27)
1: per_game-opponent (2020): (13, 27)
2: totals-team (2020): (13, 27)
3: totals-opponent (2020): (13, 27)
4: advanced-team (2020): (14, 31)
5: per_poss-team (2020): (12, 27)
6: per_poss-opponent (2020): (12, 27)
7: shooting-team (2020): (13, 28)
8: shooting-opponent (2020): (13, 28)
9: per_game-team (2021): (13, 27)
10: per_game-opponent (2021): (13, 27)
11: totals-team (2021): (13, 27)
12: totals-opponent (2021): (13, 27)
13: advanced-team (2021): (14, 31)
14: per_poss-team (2021): (12, 27)
15: per_poss-opponent (2021): (12, 27)
16: shooting-team (2021): (13, 28)
17: shooting-opponent (2021): (13, 28)
18: per_game-team (2022): (13, 27)
19: per_game-opponent (2022): (13, 27)
20: totals-team (2022): (13, 27)
21: totals-opponent (2022): (13, 27)
22: advanced-team (2022): (14, 31)
23: per_poss-team (2022): (12, 27)
24: per_poss-opponent (2022): (12, 27)
25: shooting-team (2022): (13, 28)
26: shooting-opponent (2022): (13, 28)
27: per_game-team (2023): (13

In [13]:
# group tables by type and stack each group
from collections import defaultdict

# collect tables by type
grouped_tables = defaultdict(list)
for df in all_tables:
    grouped_tables[df["table_id"].iloc[0]].append(df)

# stack each type
stacked_tables = {}
for table_id, dfs in grouped_tables.items():
    stacked_tables[table_id] = pd.concat(dfs, ignore_index=True)

# preview available keys and shapes
for k, v in stacked_tables.items():
    print(f"{k}: {v.shape}")

per_game-team: (52, 27)
per_game-opponent: (52, 27)
totals-team: (52, 27)
totals-opponent: (52, 27)
advanced-team: (56, 31)
per_poss-team: (48, 27)
per_poss-opponent: (48, 27)
shooting-team: (52, 28)
shooting-opponent: (52, 28)


In [16]:
advanced_team_df = stacked_tables["advanced-team"]

advanced_team_df

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Offense Four Factors.3,Unnamed: 22,Defense Four Factors,Defense Four Factors.1,Defense Four Factors.2,Defense Four Factors.3,Unnamed: 27,Unnamed: 28,table_id,year
0,Rk,Team,Age,W,L,PW,PL,MOV,SOS,SRS,...,FT/FGA,NaN,eFG%,TOV%,DRB%,FT/FGA,NaN,Arena,advanced-team,2020
1,1,Seattle Storm*,27.8,18,4,19,3,11.55,-0.96,10.58,...,.227,NaN,.454,16.7,74.0,.202,NaN,IMG Academy,advanced-team,2020
2,2,Las Vegas Aces*,27.1,18,4,18,4,8.59,-0.72,7.88,...,.277,NaN,.485,14.9,78.9,.157,NaN,IMG Academy,advanced-team,2020
3,3,Los Angeles Sparks*,28.6,15,7,15,7,4.59,-0.38,4.21,...,.199,NaN,.511,18.9,76.0,.204,NaN,IMG Academy,advanced-team,2020
4,4,Minnesota Lynx*,24.8,14,8,14,8,3.82,-0.32,3.50,...,.223,NaN,.505,16.3,75.1,.217,NaN,IMG Academy,advanced-team,2020
5,5,Chicago Sky*,27.2,12,10,13,9,2.64,-0.22,2.42,...,.178,NaN,.503,15.0,76.4,.205,NaN,IMG Academy,advanced-team,2020
6,6,Phoenix Mercury*,28.0,13,9,13,9,2.00,-0.17,1.83,...,.259,NaN,.477,14.2,73.1,.233,NaN,IMG Academy,advanced-team,2020
7,7,Connecticut Sun*,27.5,10,12,11,11,0.45,-0.04,0.42,...,.217,NaN,.495,16.9,78.4,.238,NaN,IMG Academy,advanced-team,2020
8,8,Washington Mystics*,27.1,9,13,10,12,-1.55,0.13,-1.42,...,.181,NaN,.524,16.4,76.1,.217,NaN,IMG Academy,advanced-team,2020
9,9,Dallas Wings,23.8,8,14,8,14,-3.55,0.30,-3.25,...,.220,NaN,.521,15.1,73.4,.257,NaN,IMG Academy,advanced-team,2020


In [30]:
# flatten advanced_team_df using row 0 as header
advanced_team_df.columns = advanced_team_df.iloc[0]
advanced_team_df = advanced_team_df[1:].reset_index(drop=True)

In [31]:
advanced_team_df

,Rk,Team,Age,W,L,PW,PL,MOV,SOS,SRS,...,FT/FGA,NaN,eFG%,TOV%,DRB%,FT/FGA,NaN,Arena,advanced-team,2020
0,1,Seattle Storm*,27.8,18,4,19,3,11.55,-0.96,10.58,...,.227,NaN,.454,16.7,74.0,.202,NaN,IMG Academy,advanced-team,2020
1,2,Las Vegas Aces*,27.1,18,4,18,4,8.59,-0.72,7.88,...,.277,NaN,.485,14.9,78.9,.157,NaN,IMG Academy,advanced-team,2020
2,3,Los Angeles Sparks*,28.6,15,7,15,7,4.59,-0.38,4.21,...,.199,NaN,.511,18.9,76.0,.204,NaN,IMG Academy,advanced-team,2020
3,4,Minnesota Lynx*,24.8,14,8,14,8,3.82,-0.32,3.50,...,.223,NaN,.505,16.3,75.1,.217,NaN,IMG Academy,advanced-team,2020
4,5,Chicago Sky*,27.2,12,10,13,9,2.64,-0.22,2.42,...,.178,NaN,.503,15.0,76.4,.205,NaN,IMG Academy,advanced-team,2020
5,6,Phoenix Mercury*,28.0,13,9,13,9,2.00,-0.17,1.83,...,.259,NaN,.477,14.2,73.1,.233,NaN,IMG Academy,advanced-team,2020
6,7,Connecticut Sun*,27.5,10,12,11,11,0.45,-0.04,0.42,...,.217,NaN,.495,16.9,78.4,.238,NaN,IMG Academy,advanced-team,2020
7,8,Washington Mystics*,27.1,9,13,10,12,-1.55,0.13,-1.42,...,.181,NaN,.524,16.4,76.1,.217,NaN,IMG Academy,advanced-team,2020
8,9,Dallas Wings,23.8,8,14,8,14,-3.55,0.30,-3.25,...,.220,NaN,.521,15.1,73.4,.257,NaN,IMG Academy,advanced-team,2020
9,10,Atlanta Dream,25.8,7,15,5,17,-6.68,0.56,-6.13,...,.172,NaN,.513,14.1,75.7,.231,NaN,IMG Academy,advanced-team,2020


In [32]:
# drop rows where Team == 'League Average'
advanced_team_df = advanced_team_df[advanced_team_df["Team"] != "League Average"]

# drop the 'Rk' column if it exists
if "Rk" in advanced_team_df.columns:
    advanced_team_df = advanced_team_df.drop(columns=["Rk"])

In [35]:
for col in advanced_team_df.columns.to_list():
    null_count = advanced_team_df[col].isnull().sum()
    if null_count > 0:
        print(f"{repr(col)}: {null_count}")

TypeError: only integer scalar arrays can be converted to a scalar index

In [40]:
# create one df per table type using dictionary-style lookup
per_game_team_df = stacked_tables["per_game-team"]
per_game_opp_df = stacked_tables["per_game-opponent"]
totals_team_df = stacked_tables["totals-team"]
totals_opp_df = stacked_tables["totals-opponent"]
per_poss_team_df = stacked_tables["per_poss-team"]
per_poss_opp_df = stacked_tables["per_poss-opponent"]
shooting_team_df = stacked_tables["shooting-team"]
shooting_opp_df = stacked_tables["shooting-opponent"]

In [41]:
# drop 'Rk' column from each non-advanced DataFrame (if it exists)
for df in [
    per_game_team_df, per_game_opp_df,
    totals_team_df, totals_opp_df,
    per_poss_team_df, per_poss_opp_df,
    shooting_team_df, shooting_opp_df
]:
    if "Rk" in df.columns:
        df.drop(columns=["Rk"], inplace=True)

In [42]:
per_game_team_df = stacked_tables["per_game-team"]
per_game_opp_df = stacked_tables["per_game-opponent"]
totals_team_df = stacked_tables["totals-team"]
totals_opp_df = stacked_tables["totals-opponent"]
per_poss_team_df = stacked_tables["per_poss-team"]
per_poss_opp_df = stacked_tables["per_poss-opponent"]
shooting_team_df = stacked_tables["shooting-team"]
shooting_opp_df = stacked_tables["shooting-opponent"]
advanced_team_df = stacked_tables["advanced-team"]

In [44]:
# prefix all columns except 'Team' and 'year'
def prefix_cols(df, prefix):
    return df.rename(columns={col: f"{prefix}_{col}" for col in df.columns if col not in {"Team", "year"}})

# apply to each DataFrame
per_game_team_df     = prefix_cols(per_game_team_df, "team_per_game")
per_game_opp_df      = prefix_cols(per_game_opp_df, "opp_per_game")
totals_team_df       = prefix_cols(totals_team_df, "team_totals")
totals_opp_df        = prefix_cols(totals_opp_df, "opp_totals")
per_poss_team_df     = prefix_cols(per_poss_team_df, "team_per_poss")
per_poss_opp_df      = prefix_cols(per_poss_opp_df, "opp_per_poss")
shooting_team_df     = prefix_cols(shooting_team_df, "team_shooting")
shooting_opp_df      = prefix_cols(shooting_opp_df, "opp_shooting")
advanced_team_df     = prefix_cols(advanced_team_df, "team_advanced")

In [46]:
dfs = {
    "per_game_team_df": per_game_team_df,
    "per_game_opp_df": per_game_opp_df,
    "totals_team_df": totals_team_df,
    "totals_opp_df": totals_opp_df,
    "per_poss_team_df": per_poss_team_df,
    "per_poss_opp_df": per_poss_opp_df,
    "shooting_team_df": shooting_team_df,
    "shooting_opp_df": shooting_opp_df,
    "advanced_team_df": advanced_team_df,
}

for name, df in dfs.items():
    print(name, "has 'Team'? →", 'Team' in df.columns)

per_game_team_df has 'Team'? → True
per_game_opp_df has 'Team'? → True
totals_team_df has 'Team'? → True
totals_opp_df has 'Team'? → True
per_poss_team_df has 'Team'? → True
per_poss_opp_df has 'Team'? → True
shooting_team_df has 'Team'? → False
shooting_opp_df has 'Team'? → False
advanced_team_df has 'Team'? → True


In [62]:
shooting_team_df = stacked_tables["shooting-team"]
shooting_opp_df = stacked_tables["shooting-opponent"]

In [63]:
print(shooting_team_df.columns.to_list())
shooting_team_df.head(1)

['level_0_Rk', 'level_0_Team', 'level_0_G', 'level_0_MP', 'level_0_FG%', 'level_0_Dist.', 'level_0_Unnamed: 6_level_1', '% of FGA by Distance_2P', '% of FGA by Distance_0-3', '% of FGA by Distance_3-10', '% of FGA by Distance_10-16', '% of FGA by Distance_16-3P', '% of FGA by Distance_3P', 'level_0_Unnamed: 13_level_1', 'FG% by Distance_2P', 'FG% by Distance_0-3', 'FG% by Distance_3-10', 'FG% by Distance_10-16', 'FG% by Distance_16-3P', 'FG% by Distance_3P', 'level_0_Unnamed: 20_level_1', "% of FG Ast'd_2P", "% of FG Ast'd_3P", 'level_0_Unnamed: 23_level_1', 'Corner_%3PA', 'Corner_3P%', 'table_id', 'year']


,level_0_Rk,level_0_Team,level_0_G,level_0_MP,level_0_FG%,level_0_Dist.,level_0_Unnamed: 6_level_1,% of FGA by Distance_2P,% of FGA by Distance_0-3,% of FGA by Distance_3-10,...,FG% by Distance_16-3P,FG% by Distance_3P,level_0_Unnamed: 20_level_1,% of FG Ast'd_2P,% of FG Ast'd_3P,level_0_Unnamed: 23_level_1,Corner_%3PA,Corner_3P%,table_id,year
0,1.0,Atlanta Dream,22,4450,0.442,13.1,NaN,0.763,0.13,0.311,...,0.399,0.35,NaN,0.472,0.815,NaN,0.094,0.343,shooting-team,2020


In [64]:
print(shooting_opp_df.columns.to_list())
shooting_opp_df.head(1)

['level_0_Rk', 'level_0_Team', 'level_0_G', 'level_0_MP', 'level_0_FG%', 'level_0_Dist.', 'level_0_Unnamed: 6_level_1', '% of FGA by Distance_2P', '% of FGA by Distance_0-3', '% of FGA by Distance_3-10', '% of FGA by Distance_10-16', '% of FGA by Distance_16-3P', '% of FGA by Distance_3P', 'level_0_Unnamed: 13_level_1', 'FG% by Distance_2P', 'FG% by Distance_0-3', 'FG% by Distance_3-10', 'FG% by Distance_10-16', 'FG% by Distance_16-3P', 'FG% by Distance_3P', 'level_0_Unnamed: 20_level_1', "% of FG Ast'd_2P", "% of FG Ast'd_3P", 'level_0_Unnamed: 23_level_1', 'Corner_%3PA', 'Corner_3P%', 'table_id', 'year']


,level_0_Rk,level_0_Team,level_0_G,level_0_MP,level_0_FG%,level_0_Dist.,level_0_Unnamed: 6_level_1,% of FGA by Distance_2P,% of FGA by Distance_0-3,% of FGA by Distance_3-10,...,FG% by Distance_16-3P,FG% by Distance_3P,level_0_Unnamed: 20_level_1,% of FG Ast'd_2P,% of FG Ast'd_3P,level_0_Unnamed: 23_level_1,Corner_%3PA,Corner_3P%,table_id,year
0,1.0,Atlanta Dream,22,4450,0.457,14.0,NaN,0.674,0.121,0.321,...,0.403,0.348,NaN,0.549,0.879,NaN,0.114,0.421,shooting-opponent,2020


In [65]:
# For team shooting
shooting_team_df.columns = [
    col.split("_", 1)[-1] if col.startswith("Unnamed:") else col
    for col in shooting_team_df.columns
]

# For opponent shooting
shooting_opp_df.columns = [
    col.split("_", 1)[-1] if col.startswith("Unnamed:") else col
    for col in shooting_opp_df.columns
]

In [67]:
def clean_columns(df):
    df.columns = [
        re.sub(r'^(Unnamed: \d+_level_0_|Unnamed: \d+_level_1_|level_0_)', '', col)
        for col in df.columns
    ]
    return df

# Apply to both shooting tables
shooting_team_df = clean_columns(shooting_team_df)
shooting_opp_df = clean_columns(shooting_opp_df)


In [68]:
shooting_team_df

,Rk,Team,G,MP,FG%,Dist.,Unnamed: 6_level_1,% of FGA by Distance_2P,% of FGA by Distance_0-3,% of FGA by Distance_3-10,...,FG% by Distance_16-3P,FG% by Distance_3P,Unnamed: 20_level_1,% of FG Ast'd_2P,% of FG Ast'd_3P,Unnamed: 23_level_1,Corner_%3PA,Corner_3P%,table_id,year
0,1.0,Atlanta Dream,22,4450,0.442,13.1,NaN,0.763,0.130,0.311,...,0.399,0.350,NaN,0.472,0.815,NaN,0.094,0.343,shooting-team,2020
1,2.0,Chicago Sky*,22,4400,0.491,13.6,NaN,0.683,0.137,0.305,...,0.497,0.351,NaN,0.579,0.898,NaN,0.118,0.411,shooting-team,2020
2,3.0,Connecticut Sun*,22,4425,0.427,12.7,NaN,0.727,0.145,0.354,...,0.371,0.311,NaN,0.562,0.838,NaN,0.100,0.405,shooting-team,2020
3,4.0,Dallas Wings,22,4450,0.415,15.0,NaN,0.611,0.100,0.300,...,0.324,0.323,NaN,0.409,0.811,NaN,0.105,0.344,shooting-team,2020
4,5.0,Indiana Fever,22,4400,0.441,13.8,NaN,0.700,0.126,0.296,...,0.374,0.345,NaN,0.593,0.842,NaN,0.102,0.333,shooting-team,2020
5,6.0,Los Angeles Sparks*,22,4450,0.481,13.3,NaN,0.729,0.148,0.297,...,0.421,0.398,NaN,0.545,0.887,NaN,0.160,0.438,shooting-team,2020
6,7.0,Las Vegas Aces*,22,4400,0.477,11.7,NaN,0.833,0.138,0.376,...,0.449,0.368,NaN,0.595,0.903,NaN,0.107,0.407,shooting-team,2020
7,8.0,Minnesota Lynx*,22,4400,0.456,13.7,NaN,0.679,0.138,0.310,...,0.346,0.385,NaN,0.542,0.929,NaN,0.089,0.452,shooting-team,2020
8,9.0,New York Liberty,22,4400,0.372,14.8,NaN,0.585,0.121,0.305,...,0.305,0.277,NaN,0.499,0.844,NaN,0.113,0.191,shooting-team,2020
9,10.0,Phoenix Mercury*,22,4425,0.450,14.5,NaN,0.634,0.113,0.308,...,0.388,0.343,NaN,0.647,0.746,NaN,0.104,0.357,shooting-team,2020


In [69]:
shooting_opp_df

,Rk,Team,G,MP,FG%,Dist.,Unnamed: 6_level_1,% of FGA by Distance_2P,% of FGA by Distance_0-3,% of FGA by Distance_3-10,...,FG% by Distance_16-3P,FG% by Distance_3P,Unnamed: 20_level_1,% of FG Ast'd_2P,% of FG Ast'd_3P,Unnamed: 23_level_1,Corner_%3PA,Corner_3P%,table_id,year
0,1.0,Atlanta Dream,22,4450,0.457,14.0,NaN,0.674,0.121,0.321,...,0.403,0.348,NaN,0.549,0.879,NaN,0.114,0.421,shooting-opponent,2020
1,2.0,Chicago Sky*,22,4400,0.454,13.0,NaN,0.732,0.158,0.321,...,0.432,0.368,NaN,0.555,0.861,NaN,0.098,0.400,shooting-opponent,2020
2,3.0,Connecticut Sun*,22,4425,0.443,14.0,NaN,0.682,0.143,0.316,...,0.385,0.330,NaN,0.564,0.880,NaN,0.101,0.348,shooting-opponent,2020
3,4.0,Dallas Wings,22,4450,0.471,13.0,NaN,0.731,0.142,0.374,...,0.396,0.371,NaN,0.594,0.857,NaN,0.124,0.449,shooting-opponent,2020
4,5.0,Indiana Fever,22,4400,0.472,13.0,NaN,0.734,0.153,0.335,...,0.372,0.370,NaN,0.526,0.827,NaN,0.104,0.429,shooting-opponent,2020
5,6.0,Los Angeles Sparks*,22,4450,0.449,14.0,NaN,0.667,0.131,0.284,...,0.378,0.373,NaN,0.611,0.860,NaN,0.104,0.400,shooting-opponent,2020
6,7.0,Las Vegas Aces*,22,4400,0.431,14.0,NaN,0.650,0.110,0.302,...,0.418,0.311,NaN,0.562,0.829,NaN,0.117,0.313,shooting-opponent,2020
7,8.0,Minnesota Lynx*,22,4400,0.446,14.0,NaN,0.647,0.122,0.287,...,0.382,0.331,NaN,0.597,0.882,NaN,0.149,0.408,shooting-opponent,2020
8,9.0,New York Liberty,22,4400,0.444,14.0,NaN,0.723,0.117,0.305,...,0.412,0.344,NaN,0.514,0.815,NaN,0.084,0.405,shooting-opponent,2020
9,10.0,Phoenix Mercury*,22,4425,0.425,13.0,NaN,0.700,0.123,0.330,...,0.396,0.344,NaN,0.520,0.876,NaN,0.103,0.313,shooting-opponent,2020


In [71]:
dfs = {
    "per_game_team_df": per_game_team_df,
    "per_game_opp_df": per_game_opp_df,
    "totals_team_df": totals_team_df,
    "totals_opp_df": totals_opp_df,
    "per_poss_team_df": per_poss_team_df,
    "per_poss_opp_df": per_poss_opp_df,
    "shooting_team_df": shooting_team_df,
    "shooting_opp_df": shooting_opp_df,
    "advanced_team_df": advanced_team_df,
}

for name, df in dfs.items():
    print(name, "has 'year'? →", 'year' in df.columns)

per_game_team_df has 'year'? → True
per_game_opp_df has 'year'? → True
totals_team_df has 'year'? → True
totals_opp_df has 'year'? → True
per_poss_team_df has 'year'? → True
per_poss_opp_df has 'year'? → True
shooting_team_df has 'year'? → True
shooting_opp_df has 'year'? → True
advanced_team_df has 'year'? → False


In [72]:
advanced_team_df

,Team,team_advanced_Age,team_advanced_W,team_advanced_L,team_advanced_PW,team_advanced_PL,team_advanced_MOV,team_advanced_SOS,team_advanced_SRS,team_advanced_ORtg,...,team_advanced_FT/FGA,team_advanced_nan,team_advanced_eFG%,team_advanced_TOV%,team_advanced_DRB%,team_advanced_FT/FGA,team_advanced_nan,team_advanced_Arena,team_advanced_advanced-team,team_advanced_2020
0,Seattle Storm*,27.8,18,4,19,3,11.55,-0.96,10.58,110.4,...,.227,NaN,.454,16.7,74.0,.202,NaN,IMG Academy,advanced-team,2020
1,Las Vegas Aces*,27.1,18,4,18,4,8.59,-0.72,7.88,109.6,...,.277,NaN,.485,14.9,78.9,.157,NaN,IMG Academy,advanced-team,2020
2,Los Angeles Sparks*,28.6,15,7,15,7,4.59,-0.38,4.21,105.9,...,.199,NaN,.511,18.9,76.0,.204,NaN,IMG Academy,advanced-team,2020
3,Minnesota Lynx*,24.8,14,8,14,8,3.82,-0.32,3.50,109.6,...,.223,NaN,.505,16.3,75.1,.217,NaN,IMG Academy,advanced-team,2020
4,Chicago Sky*,27.2,12,10,13,9,2.64,-0.22,2.42,107.6,...,.178,NaN,.503,15.0,76.4,.205,NaN,IMG Academy,advanced-team,2020
5,Phoenix Mercury*,28.0,13,9,13,9,2.00,-0.17,1.83,106.8,...,.259,NaN,.477,14.2,73.1,.233,NaN,IMG Academy,advanced-team,2020
6,Connecticut Sun*,27.5,10,12,11,11,0.45,-0.04,0.42,101.7,...,.217,NaN,.495,16.9,78.4,.238,NaN,IMG Academy,advanced-team,2020
7,Washington Mystics*,27.1,9,13,10,12,-1.55,0.13,-1.42,103.5,...,.181,NaN,.524,16.4,76.1,.217,NaN,IMG Academy,advanced-team,2020
8,Dallas Wings,23.8,8,14,8,14,-3.55,0.30,-3.25,105.9,...,.220,NaN,.521,15.1,73.4,.257,NaN,IMG Academy,advanced-team,2020
9,Atlanta Dream,25.8,7,15,5,17,-6.68,0.56,-6.13,100.1,...,.172,NaN,.513,14.1,75.7,.231,NaN,IMG Academy,advanced-team,2020


In [73]:
# rename last column to 'year'
advanced_team_df.columns.values[-1] = "year"

In [76]:
# remove '*' from team names
for df in [
    per_game_team_df,
    per_game_opp_df,
    totals_team_df,
    totals_opp_df,
    per_poss_team_df,
    per_poss_opp_df,
    shooting_team_df,
    shooting_opp_df,
    advanced_team_df
]:
    if "Team" in df.columns:
        df["Team"] = df["Team"].str.replace("*", "", regex=False).str.strip()

In [77]:
full_team_df = []

full_team_df = per_game_team_df.merge(per_game_opp_df, on=["Team", "year"], how="outer")
full_team_df = full_team_df.merge(totals_team_df, on=["Team", "year"], how="outer")
full_team_df = full_team_df.merge(totals_opp_df, on=["Team", "year"], how="outer")
full_team_df = full_team_df.merge(per_poss_team_df, on=["Team", "year"], how="outer")
full_team_df = full_team_df.merge(per_poss_opp_df, on=["Team", "year"], how="outer")
full_team_df = full_team_df.merge(shooting_team_df, on=["Team", "year"], how="outer")
full_team_df = full_team_df.merge(shooting_opp_df, on=["Team", "year"], how="outer")
full_team_df = full_team_df.merge(advanced_team_df, on=["Team", "year"], how="outer")

In [78]:
full_team_df

,Team,team_per_game_G,team_per_game_MP,team_per_game_FG,team_per_game_FGA,team_per_game_FG%,team_per_game_3P,team_per_game_3PA,team_per_game_3P%,team_per_game_2P,...,team_advanced_ORB%,team_advanced_FT/FGA,team_advanced_nan,team_advanced_eFG%,team_advanced_TOV%,team_advanced_DRB%,team_advanced_FT/FGA,team_advanced_nan,team_advanced_Arena,team_advanced_advanced-team
0,Atlanta Dream,22.0,202.3,31.4,71.0,0.442,5.9,16.9,0.350,25.5,...,25.3,.172,NaN,.513,14.1,75.7,.231,NaN,IMG Academy,advanced-team
1,Atlanta Dream,32.0,201.6,30.6,73.5,0.417,6.1,19.8,0.310,24.5,...,24.6,.154,NaN,.520,16.8,73.7,.234,NaN,Gateway Center,advanced-team
2,Atlanta Dream,36.0,201.4,28.8,68.4,0.420,7.5,21.4,0.351,21.2,...,24.5,.197,NaN,.494,15.7,78.5,.240,NaN,Gateway Center Arena @ College Park,advanced-team
3,Atlanta Dream,40.0,201.3,29.4,68.7,0.428,6.4,19.2,0.336,23.0,...,22.4,.252,NaN,.482,14.2,78.2,.251,NaN,Gateway Center Arena @ College Park,advanced-team
4,Chicago Sky,22.0,200.0,33.5,68.2,0.491,7.6,21.6,0.351,25.9,...,23.3,.178,NaN,.503,15.0,76.4,.205,NaN,IMG Academy,advanced-team
5,Chicago Sky,32.0,203.1,30.8,69.9,0.441,7.3,21.2,0.343,23.5,...,23.8,.205,NaN,.480,15.9,73.9,.212,NaN,Wintrust Arena,advanced-team
6,Chicago Sky,36.0,202.1,32.7,67.9,0.481,7.2,20.9,0.345,25.4,...,22.7,.202,NaN,.483,14.4,76.2,.176,NaN,Wintrust Arena,advanced-team
7,Chicago Sky,40.0,201.3,30.7,69.6,0.442,8.3,22.2,0.372,22.5,...,24.4,.173,NaN,.499,14.6,74.6,.216,NaN,Wintrust Arena,advanced-team
8,Connecticut Sun,22.0,201.1,29.7,69.5,0.427,5.9,19.0,0.311,23.8,...,27.8,.217,NaN,.495,16.9,78.4,.238,NaN,IMG Academy,advanced-team
9,Connecticut Sun,32.0,201.6,29.2,65.6,0.444,7.0,19.6,0.356,22.2,...,31.2,.220,NaN,.459,16.1,82.1,.201,NaN,Mohegan Sun Arena,advanced-team


In [79]:
# save to CSV
full_team_df.to_csv("uncleaned_full_team_df.csv", index=False)

# download to local machine
from google.colab import files
files.download("uncleaned_full_team_df.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# load the player game logs CSV from the data folder
df = pd.read_csv("LHL-final-final-project/data/uncleaned_full_team_df.csv")


# preview
df.head()

,Team,team_per_game_G,team_per_game_MP,team_per_game_FG,team_per_game_FGA,team_per_game_FG%,team_per_game_3P,team_per_game_3PA,team_per_game_3P%,team_per_game_2P,...,team_advanced_ORB%,team_advanced_FT/FGA,team_advanced_nan.1,team_advanced_eFG%.1,team_advanced_TOV%.1,team_advanced_DRB%,team_advanced_FT/FGA.1,team_advanced_nan.2,team_advanced_Arena,team_advanced_advanced-team
0,Atlanta Dream,22.0,202.3,31.4,71.0,0.442,5.9,16.9,0.350,25.5,...,25.3,.172,NaN,.513,14.1,75.7,.231,NaN,IMG Academy,advanced-team
1,Atlanta Dream,32.0,201.6,30.6,73.5,0.417,6.1,19.8,0.310,24.5,...,24.6,.154,NaN,.520,16.8,73.7,.234,NaN,Gateway Center,advanced-team
2,Atlanta Dream,36.0,201.4,28.8,68.4,0.420,7.5,21.4,0.351,21.2,...,24.5,.197,NaN,.494,15.7,78.5,.240,NaN,Gateway Center Arena @ College Park,advanced-team
3,Atlanta Dream,40.0,201.3,29.4,68.7,0.428,6.4,19.2,0.336,23.0,...,22.4,.252,NaN,.482,14.2,78.2,.251,NaN,Gateway Center Arena @ College Park,advanced-team
4,Chicago Sky,22.0,200.0,33.5,68.2,0.491,7.6,21.6,0.351,25.9,...,23.3,.178,NaN,.503,15.0,76.4,.205,NaN,IMG Academy,advanced-team


In [4]:
df.shape

(55, 226)

In [5]:
# show columns with nulls, sorted descending by null count
df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False)

,0
team_advanced_nan,55
Unnamed: 23_level_1_x,55
Unnamed: 6_level_1_y,55
Unnamed: 13_level_1_y,55
Unnamed: 20_level_1_y,55
...,...
% of FGA by Distance_10-16_y,3
% of FGA by Distance_2P_y,3
Corner_%3PA_y,3
Corner_3P%_y,3


In [6]:
# drop columns with exactly 55 nulls
df = df.drop(columns=df.columns[df.isnull().sum() == 55])

In [7]:
# show columns with nulls, sorted descending by null count
df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False)

,0
opp_per_game_STL,7
opp_per_game_TOV,7
opp_per_game_BLK,7
opp_per_game_PTS,7
opp_per_game_PF,7
...,...
FG% by Distance_16-3P_y,3
Corner_3P%_y,3
Corner_%3PA_y,3
table_id_y,3


In [8]:
# display rows with nulls in column 'opp_per_game_STL'
df[df['opp_per_game_STL'].isnull()]

,Team,team_per_game_G,team_per_game_MP,team_per_game_FG,team_per_game_FGA,team_per_game_FG%,team_per_game_3P,team_per_game_3PA,team_per_game_3P%,team_per_game_2P,...,team_advanced_eFG%,team_advanced_TOV%,team_advanced_ORB%,team_advanced_FT/FGA,team_advanced_eFG%.1,team_advanced_TOV%.1,team_advanced_DRB%,team_advanced_FT/FGA.1,team_advanced_Arena,team_advanced_advanced-team
24,League Average,22.0,200.9,30.4,68.2,0.446,7.3,21.2,0.345,23.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,League Average,32.0,201.7,29.7,68.4,0.435,7.3,21.2,0.343,22.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,League Average,36.0,201.4,30.1,68.0,0.442,7.7,22.4,0.346,22.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,League Average,40.0,201.0,30.1,68.3,0.441,7.7,22.1,0.347,22.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48,Team,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,eFG%,TOV%,ORB%,FT/FGA,eFG%,TOV%,DRB%,FT/FGA,Arena,advanced-team
49,Team,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,eFG%,TOV%,ORB%,FT/FGA,eFG%,TOV%,DRB%,FT/FGA,Arena,advanced-team
50,Team,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,eFG%,TOV%,ORB%,FT/FGA,eFG%,TOV%,DRB%,FT/FGA,Arena,advanced-team


In [9]:
# drop rows where 'Team' is "League Average" or "Team"
df = df[~df['Team'].isin(["League Average", "Team"])]

In [10]:
# show columns with nulls, sorted descending by null count
df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False)

,0


In [11]:
for col in df.columns:
  print(col)

Team
team_per_game_G
team_per_game_MP
team_per_game_FG
team_per_game_FGA
team_per_game_FG%
team_per_game_3P
team_per_game_3PA
team_per_game_3P%
team_per_game_2P
team_per_game_2PA
team_per_game_2P%
team_per_game_FT
team_per_game_FTA
team_per_game_FT%
team_per_game_ORB
team_per_game_DRB
team_per_game_TRB
team_per_game_AST
team_per_game_STL
team_per_game_BLK
team_per_game_TOV
team_per_game_PF
team_per_game_PTS
team_per_game_table_id
year
opp_per_game_G
opp_per_game_MP
opp_per_game_FG
opp_per_game_FGA
opp_per_game_FG%
opp_per_game_3P
opp_per_game_3PA
opp_per_game_3P%
opp_per_game_2P
opp_per_game_2PA
opp_per_game_2P%
opp_per_game_FT
opp_per_game_FTA
opp_per_game_FT%
opp_per_game_ORB
opp_per_game_DRB
opp_per_game_TRB
opp_per_game_AST
opp_per_game_STL
opp_per_game_BLK
opp_per_game_TOV
opp_per_game_PF
opp_per_game_PTS
opp_per_game_table_id
team_totals_G
team_totals_MP
team_totals_FG
team_totals_FGA
team_totals_FG%
team_totals_3P
team_totals_3PA
team_totals_3P%
team_totals_2P
team_totals_2PA
te

In [12]:
# drop listed columns from df
columns_to_drop = [
    "team_advanced_advanced-team", "team_advanced_W", "team_advanced_L", "team_advanced_PW", "team_advanced_PL",
    "team_per_game_G", "team_per_game_MP", "opp_per_game_G", "opp_per_game_MP", "opp_per_game_table_id",
    "team_totals_G", "team_totals_MP", "team_totals_FG", "team_totals_FGA", "team_totals_FG%", "team_totals_3P",
    "team_totals_3PA", "team_totals_3P%", "team_totals_2P", "team_totals_2PA", "team_totals_2P%", "team_totals_FT",
    "team_totals_FTA", "team_totals_FT%", "team_totals_ORB", "team_totals_DRB", "team_totals_TRB", "team_totals_AST",
    "team_totals_STL", "team_totals_BLK", "team_totals_TOV", "team_totals_PF", "team_totals_PTS", "team_totals_table_id",
    "opp_totals_G", "opp_totals_MP", "opp_totals_FG", "opp_totals_FGA", "opp_totals_FG%", "opp_totals_3P",
    "opp_totals_3PA", "opp_totals_3P%", "opp_totals_2P", "opp_totals_2PA", "opp_totals_2P%", "opp_totals_FT",
    "opp_totals_FTA", "opp_totals_FT%", "opp_totals_ORB", "opp_totals_DRB", "opp_totals_TRB", "opp_totals_AST",
    "opp_totals_STL", "opp_totals_BLK", "opp_totals_TOV", "opp_totals_PF", "opp_totals_PTS", "opp_totals_table_id",
    "team_per_poss_G", "team_per_poss_MP", "team_per_poss_table_id",
    "opp_per_poss_G", "opp_per_poss_MP", "opp_per_poss_table_id",
    "Rk_y", "G_y", "MP_y", "Rk_x", "table_id_x", "table_id_y"
]

df = df.drop(columns=columns_to_drop)

In [13]:
for col in df.columns:
  print(col)

Team
team_per_game_FG
team_per_game_FGA
team_per_game_FG%
team_per_game_3P
team_per_game_3PA
team_per_game_3P%
team_per_game_2P
team_per_game_2PA
team_per_game_2P%
team_per_game_FT
team_per_game_FTA
team_per_game_FT%
team_per_game_ORB
team_per_game_DRB
team_per_game_TRB
team_per_game_AST
team_per_game_STL
team_per_game_BLK
team_per_game_TOV
team_per_game_PF
team_per_game_PTS
team_per_game_table_id
year
opp_per_game_FG
opp_per_game_FGA
opp_per_game_FG%
opp_per_game_3P
opp_per_game_3PA
opp_per_game_3P%
opp_per_game_2P
opp_per_game_2PA
opp_per_game_2P%
opp_per_game_FT
opp_per_game_FTA
opp_per_game_FT%
opp_per_game_ORB
opp_per_game_DRB
opp_per_game_TRB
opp_per_game_AST
opp_per_game_STL
opp_per_game_BLK
opp_per_game_TOV
opp_per_game_PF
opp_per_game_PTS
team_per_poss_FG
team_per_poss_FGA
team_per_poss_FG%
team_per_poss_3P
team_per_poss_3PA
team_per_poss_3P%
team_per_poss_2P
team_per_poss_2PA
team_per_poss_2P%
team_per_poss_FT
team_per_poss_FTA
team_per_poss_FT%
team_per_poss_ORB
team_per_pos

In [14]:
# rename '_x' columns with 'team_' prefix and '_y' columns with 'opp_' prefix
df = df.rename(columns=lambda x: f"team_{x[:-2]}" if x.endswith("_x")
                         else f"opp_{x[:-2]}" if x.endswith("_y")
                         else x)

In [15]:
for col in df.columns:
  print(col)

Team
team_per_game_FG
team_per_game_FGA
team_per_game_FG%
team_per_game_3P
team_per_game_3PA
team_per_game_3P%
team_per_game_2P
team_per_game_2PA
team_per_game_2P%
team_per_game_FT
team_per_game_FTA
team_per_game_FT%
team_per_game_ORB
team_per_game_DRB
team_per_game_TRB
team_per_game_AST
team_per_game_STL
team_per_game_BLK
team_per_game_TOV
team_per_game_PF
team_per_game_PTS
team_per_game_table_id
year
opp_per_game_FG
opp_per_game_FGA
opp_per_game_FG%
opp_per_game_3P
opp_per_game_3PA
opp_per_game_3P%
opp_per_game_2P
opp_per_game_2PA
opp_per_game_2P%
opp_per_game_FT
opp_per_game_FTA
opp_per_game_FT%
opp_per_game_ORB
opp_per_game_DRB
opp_per_game_TRB
opp_per_game_AST
opp_per_game_STL
opp_per_game_BLK
opp_per_game_TOV
opp_per_game_PF
opp_per_game_PTS
team_per_poss_FG
team_per_poss_FGA
team_per_poss_FG%
team_per_poss_3P
team_per_poss_3PA
team_per_poss_3P%
team_per_poss_2P
team_per_poss_2PA
team_per_poss_2P%
team_per_poss_FT
team_per_poss_FTA
team_per_poss_FT%
team_per_poss_ORB
team_per_pos

In [16]:
# remove 'team_' prefix from specified columns
df = df.rename(columns={'team_G': 'G', 'team_MP': 'MP'})

In [17]:
# load the player game logs CSV from the data folder
df2 = pd.read_csv("LHL-final-final-project/data/2024_persona_and_team_data.csv")


# preview
df2.head()

,team,opp,month,day,team_score,opp_score,elite_scorer,efficient_scorer,volume_shooter,three_point_specialist,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,ATL,LAS,5,15,92,81,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,ATL,PHO,5,18,85,88,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
2,ATL,DAL,5,21,83,78,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
3,ATL,MIN,5,26,79,92,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
4,ATL,WAS,5,29,73,67,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357


In [18]:
# remove '_advanced' from all column names containing it
df.columns = df.columns.str.replace('_advanced', '', regex=False)

In [19]:
# remove '_advanced' from all column names containing it
df2.columns = df2.columns.str.replace('_shooting', '', regex=False)

In [20]:
for col in df2.columns:
  print(col)

team
opp
month
day
team_score
opp_score
elite_scorer
efficient_scorer
volume_shooter
three_point_specialist
slasher
free_throw_generator
and_one_machine
playmaker
offensive_hub
turnover_prone
floor_general
plus_minus_driver
self_creator
rim_protector
steal_artist
defensive_anchor
glass_cleaner
offensive_rebounder
defensive_rebounder
midrange_sniper
corner_3_specialist
catch_and_shoot
stretch_big
heave_chucker
all_around_star
impact_bench
fast_break_threat
team_per_game_FG
team_per_game_FGA
team_per_game_FG%
team_per_game_3P
team_per_game_3PA
team_per_game_3P%
team_per_game_2P
team_per_game_2PA
team_per_game_2P%
team_per_game_FT
team_per_game_FTA
team_per_game_FT%
team_per_game_ORB
team_per_game_DRB
team_per_game_TRB
team_per_game_AST
team_per_game_STL
team_per_game_BLK
team_per_game_TOV
team_per_game_PF
team_per_game_PTS
team_totals_FG
team_totals_FGA
team_totals_FG%
team_totals_3P
team_totals_3PA
team_totals_3P%
team_totals_2P
team_totals_2PA
team_totals_2P%
team_totals_FT
team_totals

In [21]:
# drop specified columns from df2
cols_to_drop_df2 = [
    "team_totals_FG", "team_totals_FGA", "team_totals_FG%", "team_totals_3P", "team_totals_3PA",
    "team_totals_3P%", "team_totals_2P", "team_totals_2PA", "team_totals_2P%", "team_totals_FT",
    "team_totals_FTA", "team_totals_FT%", "team_totals_ORB", "team_totals_DRB", "team_totals_TRB",
    "team_totals_AST", "team_totals_STL", "team_totals_BLK", "team_totals_TOV", "team_totals_PF",
    "team_totals_PTS", "team_W", "team_L", "team_PW", "team_PL",
    "opp_totals_FG", "opp_totals_FGA", "opp_totals_FG%", "opp_totals_3P", "opp_totals_3PA",
    "opp_totals_3P%", "opp_totals_2P", "opp_totals_2PA", "opp_totals_2P%", "opp_totals_FT",
    "opp_totals_FTA", "opp_totals_FT%", "opp_totals_ORB", "opp_totals_DRB", "opp_totals_TRB",
    "opp_totals_AST", "opp_totals_STL", "opp_totals_BLK", "opp_totals_TOV", "opp_totals_PF",
    "opp_totals_PTS"
]

df2 = df2.drop(columns=cols_to_drop_df2)

In [22]:
# drop specified columns from df
df = df.drop(columns=["team_Arena", "G", "MP", "team_per_game_table_id"])

In [23]:
# add 'year' column with value 2024 to df2
df2["year"] = 2024

In [24]:
# lowercase all column names in df
df.columns = df.columns.str.lower()

# lowercase all column names in df2
df2.columns = df2.columns.str.lower()

In [25]:
# columns in df but not in df2
diff_df = set(df.columns) - set(df2.columns)

# columns in df2 but not in df
diff_df2 = set(df2.columns) - set(df.columns)

print("Columns in df not in df2:", diff_df)
print("Columns in df2 not in df:", diff_df2)

Columns in df not in df2: set()
Columns in df2 not in df: {'stretch_big', 'catch_and_shoot', 'defensive_anchor', 'opp_score', 'playmaker', 'opp', 'day', 'free_throw_generator', 'volume_shooter', 'corner_3_specialist', 'plus_minus_driver', 'glass_cleaner', 'turnover_prone', 'all_around_star', 'elite_scorer', 'floor_general', 'offensive_rebounder', 'midrange_sniper', 'fast_break_threat', 'heave_chucker', 'slasher', 'month', 'self_creator', 'impact_bench', 'offensive_hub', 'efficient_scorer', 'three_point_specialist', 'and_one_machine', 'rim_protector', 'defensive_rebounder', 'steal_artist', 'team_score'}


In [29]:
# save to CSV
df.to_csv("cleaned_full_team_df.csv", index=False)

# save to CSV
df2.to_csv("cleaned_2024_persona_and_team_df.csv", index=False)

# download to local machine
from google.colab import files
files.download("cleaned_2024_persona_and_team_df.csv")
files.download("cleaned_full_team_df.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
test_url = "https://www.basketball-reference.com/wnba/players/q/quiglal01w/gamelog/2021/"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(test_url, headers=headers)

# Check response status
print("Status Code:", response.status_code)

# Check if redirected
print("Final URL:", response.url)

# Quick check for Cloudflare or bot-block

print("Page snippet:", response.text[:1000])

Status Code: 200
Final URL: https://www.basketball-reference.com/wnba/players/q/quiglal01w/gamelog/2021/
Page snippet: 
<!DOCTYPE html>
<html data-version="klecko-" data-root="/home/bbr/deploy/www" lang="en" class="no-js" >
<head>
    <meta charset="utf-8">
    <meta http-equiv="x-ua-compatible" content="ie=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=2.0" />
    <link rel="dns-prefetch" href="https://cdn.ssref.net/req/202504011" />
<script>
/* https://docs.osano.com/hc/en-us/articles/22469433444372-Google-Consent-Mode-v2  */
  window.dataLayer = window.dataLayer ||[];
      function gtag(){dataLayer.push(arguments);}
      gtag('consent','default',{
        'ad_storage':'denied',
        'analytics_storage':'denied',
        'ad_user_data':'denied',
        'ad_personalization':'denied',
        'personalization_storage':'denied',
        'functionality_storage':'granted',
        'security_storage':'granted',
        'wait_for_update'

In [32]:
# define years
years = range(2020, 2024)

# player list (a-z pages)
base_url = "https://www.basketball-reference.com/wnba/players/{}/"
headers = {"User-Agent": "Mozilla/5.0"}
players = []

for letter in string.ascii_lowercase:
    response = requests.get(base_url.format(letter), headers=headers)
    time.sleep(3)  # to avoid rate limits
    soup = BeautifulSoup(response.content, "html.parser")
    player_tags = soup.find_all("p")

    for tag in player_tags:
        if any(str(year) in tag.text for year in years):
            a_tag = tag.find("a")
            name = a_tag.text.strip()
            link = a_tag["href"]
            players.append((name, link))

players = list(set(players))

In [33]:
frames = []

for name, rel_link in players:
    for year in years:
        gamelog_url = f"https://www.basketball-reference.com{rel_link.replace('.html', f'/gamelog/{year}/')}"
        response = requests.get(gamelog_url, headers=headers)
        time.sleep(5)  # to avoid rate limits
        soup = BeautifulSoup(response.content, "html.parser")
        table = soup.find("table", id="wnba_pgl_basic")

        if table is None:
            print(f"No table for {name}, {year}")
            continue

        df_player = pd.read_html(StringIO(str(table)))[0]
        df_player = df_player[df_player["Rk"] != "Rk"]

        df_player = df_player.rename(columns={
            "Unnamed: 4": "home_away",
            "Unnamed: 6": "win_margin"
        })

        age_parts = df_player["Age"].str.extract(r"(\d+)-(\d+)").astype(float)
        df_player["Age"] = round(age_parts[0] + age_parts[1] / 365, 1)

        df_player["home_away"] = df_player["home_away"].apply(lambda x: "away" if x == "@" else "home")
        df_player["win_margin"] = df_player["win_margin"].str.extract(r"\(([-+]?\d+)\)").astype(float)

        def convert_mp(val):
            if pd.isna(val):
                return np.nan
            mins, secs = map(int, val.split(":"))
            return round(mins + secs / 60, 1)

        df_player["MP"] = df_player["MP"].apply(convert_mp)

        df_player.insert(0, "Year", year)
        df_player.insert(0, "Player", name)

        frames.append(df_player)

# combine all individual player dataframes
player_df = pd.concat(frames, ignore_index=True)

No table for Madi Williams, 2020
No table for Madi Williams, 2021
No table for Madi Williams, 2022
No table for Shavonte Zellous, 2020
No table for Shavonte Zellous, 2022
No table for Shavonte Zellous, 2023
No table for Taylor Soule, 2020
No table for Taylor Soule, 2021
No table for Taylor Soule, 2022
No table for Shyla Heal, 2020
No table for Shyla Heal, 2022
No table for Shyla Heal, 2023
No table for Aisha Sheppard, 2020
No table for Aisha Sheppard, 2021
No table for Aisha Sheppard, 2023
No table for Robyn Parks, 2020
No table for Robyn Parks, 2021
No table for Robyn Parks, 2022
No table for Kadi Sissoko, 2020
No table for Kadi Sissoko, 2021
No table for Kadi Sissoko, 2022
No table for NaLyssa Smith, 2020
No table for NaLyssa Smith, 2021
No table for Brittany Boyd, 2020
No table for Brittany Boyd, 2022
No table for Brittany Boyd, 2023
No table for Jordan Horston, 2020
No table for Jordan Horston, 2021
No table for Jordan Horston, 2022
No table for Laeticia Amihere, 2020
No table for 

In [34]:
# save to CSV
player_df.to_csv("2020-2023_player_gamelogs.csv", index=False)

# download to local machine
from google.colab import files
files.download("2020-2023_player_gamelogs.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
player_df.head()

,Player,Year,Rk,Date,Age,Tm,home_away,Opp,win_margin,GS,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc
0,Tianna Hawkins,2020,1,2020-07-25,29.4,WAS,home,IND,25.0,0,...,0,0,0,0,0,0,0,0,0,-0.7
1,Tianna Hawkins,2020,2,2020-08-05,29.4,WAS,home,LVA,-6.0,0,...,0,1,1,1,2,0,1,3,7,3.4
2,Tianna Hawkins,2020,3,2020-08-07,29.4,WAS,home,NYL,-8.0,0,...,0,5,5,1,0,1,2,2,0,-2.7
3,Tianna Hawkins,2020,4,2020-08-09,29.4,WAS,home,IND,-7.0,0,...,3,7,10,0,2,0,0,3,17,15.5
4,Tianna Hawkins,2020,5,2020-08-11,29.4,WAS,home,MIN,-20.0,1,...,2,4,6,3,2,1,1,3,10,10.8


In [36]:
player_df.shape

(7106, 30)

In [31]:
# load the player game logs CSV from the data folder
player_df = pd.read_csv("LHL-final-final-project/data/2020-2023_player_gamelogs.csv")

In [5]:
# load the player data CSV from the data folder
player_data = pd.read_csv("LHL-final-final-project/data/player_data.csv")

In [6]:
player_data.shape

(1069, 123)

In [32]:
for col in player_df.columns:
  print(col)

Player
Year
Rk
Date
Age
Tm
home_away
Opp
win_margin
GS
MP
FG
FGA
FG%
3P
3PA
3P%
FT
FTA
FT%
ORB
DRB
TRB
AST
STL
BLK
TOV
PF
PTS
GmSc


In [33]:
# drop specified columns from player_df
columns_to_drop = [
    'Rk', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', 'FT', 'FTA', 'FT%',
    'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS'
]

player_df = player_df.drop(columns=columns_to_drop)

In [34]:
# Convert 'Date' column to datetime
player_df['Date'] = pd.to_datetime(player_df['Date'])

# Create 'day' and 'month' columns from 'Date'
player_df['day'] = player_df['Date'].dt.day
player_df['month'] = player_df['Date'].dt.month

In [35]:
for col in player_df.columns:
  print(col)

Player
Year
Date
Age
Tm
home_away
Opp
win_margin
GS
MP
GmSc
day
month


In [36]:
# show columns with nulls, sorted descending by null count
player_df.isnull().sum()[player_df.isnull().sum() > 0].sort_values(ascending=False)

,0


In [37]:
# Drop 'Date', rename 'Tm' to 'team', make all columns lowercase
player_df = player_df.drop(columns=['Date']).rename(columns={'Tm': 'team'})
player_df.columns = player_df.columns.str.lower()

In [38]:
for col in player_df.columns:
  print(col)

player
year
age
team
home_away
opp
win_margin
gs
mp
gmsc
day
month


In [14]:
for col in player_data.columns:
  print(col)

player
year
tm
age
g
gs
per_game_mp
per_game_fg
per_game_fga
per_game_fg_pct
per_game_3p
per_game_3pa
per_game_3p_pct
per_game_2p
per_game_2pa
per_game_2p_pct
per_game_efg_pct
per_game_ft
per_game_fta
per_game_ft_pct
per_game_orb
per_game_drb
per_game_trb
per_game_ast
per_game_stl
per_game_blk
per_game_tov
per_game_pf
per_game_pts
mp
per_minute_fg
per_minute_fga
per_minute_fg_pct
per_minute_3p
per_minute_3pa
per_minute_3p_pct
per_minute_2p
per_minute_2pa
per_minute_2p_pct
per_minute_ft
per_minute_fta
per_minute_ft_pct
per_minute_orb
per_minute_drb
per_minute_trb
per_minute_ast
per_minute_stl
per_minute_blk
per_minute_tov
per_minute_pf
per_minute_pts
per_poss_fg
per_poss_fga
per_poss_fg_pct
per_poss_3p
per_poss_3pa
per_poss_3p_pct
per_poss_2p
per_poss_2pa
per_poss_2p_pct
per_poss_ft
per_poss_fta
per_poss_ft_pct
per_poss_orb
per_poss_drb
per_poss_trb
per_poss_ast
per_poss_stl
per_poss_blk
per_poss_tov
per_poss_pf
per_poss_pts
per_poss_ortg
per_poss_drtg
advanced_per
advanced_ts_pct
advan

In [15]:
# rename 'tm' column to 'team' in player_data
player_data = player_data.rename(columns={'tm': 'team'})

In [19]:
# full set of 27 personas and their primary sorting stat
personas = {
    # Offensive
    "elite_scorer": "per_game_pts",
    "efficient_scorer": "advanced_ts_pct",
    "volume_shooter": "per_game_fga",
    "three_point_specialist": "per_game_3pa",
    "slasher": "shooting_fg_pct_by_distance_0-3",
    "free_throw_generator": "per_game_fta",
    "and_one_machine": "pbp_misc_and1",

    # Playmaking / IQ
    "playmaker": "per_game_ast",
    "offensive_hub": "advanced_ast_pct",
    "turnover_prone": "per_game_tov",
    "floor_general": "advanced_ast_pct",  # sort by ast_pct, can display tov_pct too
    "plus_minus_driver": "pbp_plus_minus_per_100_poss_on_off",
    "self_creator": "advanced_usg_pct",  # sort by usg_pct, low %astd may be inspected separately

    # Defensive
    "rim_protector": "per_game_blk",
    "steal_artist": "per_game_stl",
    "defensive_anchor": "advanced_dws",
    "glass_cleaner": "per_game_trb",
    "offensive_rebounder": "advanced_orb_pct",
    "defensive_rebounder": "advanced_drb_pct",

    # Shooting types
    "midrange_sniper": "shooting_fg_pct_by_distance_10-16",
    "corner_3_specialist": "shooting_corner_3s_3p_pct",
    "catch_and_shoot": "shooting_pct_of_fg_astd_3p",
    "stretch_big": "per_game_3pa",
    "heave_chucker": "shooting_heaves_att",

    # Misc
    "all_around_star": "advanced_ws",
    "impact_bench": "per_minute_pts",
    "fast_break_threat": "pbp_misc_pga",
}

# Dictionary to store top-10 personas for each year
top_10_personas_by_year = {}

# Loop through each year and calculate top-10 personas
for year in [2023, 2022, 2021, 2020]:
    df_year = player_data[player_data["year"] == year]

    for persona, stat in personas.items():
        top_10 = df_year.sort_values(stat, ascending=False)[["player", "team", stat]].head(10).copy()
        top_10["persona"] = persona
        top_10["year"] = year

        persona_frames.append(top_10)

# combine all results into a single DataFrame
top_10_personas = pd.concat(persona_frames, ignore_index=True)

In [21]:
top_10_personas = top_10_personas[["player", "team", "persona", "year"]]

In [23]:
# One-hot encode the 'persona' column
top_10_personas = pd.get_dummies(top_10_personas, columns=['persona'])

In [25]:
# convert boolean persona columns to integers (0 and 1)
persona_cols = top_10_personas.columns.difference(['player', 'team', 'year'])
top_10_personas[persona_cols] = top_10_personas[persona_cols].astype(int)

In [26]:
top_10_personas

,player,team,year,persona_all_around_star,persona_and_one_machine,persona_catch_and_shoot,persona_corner_3_specialist,persona_defensive_anchor,persona_defensive_rebounder,persona_efficient_scorer,...,persona_playmaker,persona_plus_minus_driver,persona_rim_protector,persona_self_creator,persona_slasher,persona_steal_artist,persona_stretch_big,persona_three_point_specialist,persona_turnover_prone,persona_volume_shooter
0,Jewell Loyd,SEA,2023,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Breanna Stewart,NYL,2023,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,A'ja Wilson,LVA,2023,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Napheesa Collier,MIN,2023,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Arike Ogunbowale,DAL,2023,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1075,Betnijah Laney-Hamilton,ATL,2020,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1076,Arike Ogunbowale,DAL,2020,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1077,Layshia Clarendon,NYL,2020,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1078,Diana Taurasi,PHO,2020,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [39]:
# Find duplicate player/year combinations
duplicates = top_10_personas.groupby(['player', 'year']).size().reset_index(name='count')
duplicates = duplicates[duplicates['count'] > 1]

duplicates.sort_values(['player', 'year'], ascending=[True, False])

,player,year,count
3,A'ja Wilson,2023,15
2,A'ja Wilson,2022,13
1,A'ja Wilson,2021,11
0,A'ja Wilson,2020,12
10,Aerial Powers,2023,2
...,...,...,...
265,Teaira McCowan,2021,5
264,Teaira McCowan,2020,3
269,Tiffany Hayes,2022,2
274,Tina Charles,2022,6


In [41]:
# First, aggregate persona columns per player/team/year
persona_cols = top_10_personas.columns.difference(['player', 'team', 'year'])
top_10_personas = top_10_personas.groupby(['player', 'team', 'year'], as_index=False)[persona_cols].max()

In [44]:
# Find duplicate player/year combinations
duplicates = top_10_personas.groupby(['player', 'team', 'year']).size().reset_index(name='count')
duplicates = duplicates[duplicates['count'] > 1]

duplicates.sort_values(['player', 'year'], ascending=[True, False])

,player,team,year,count


In [42]:
top_10_personas

,player,team,year,persona_all_around_star,persona_and_one_machine,persona_catch_and_shoot,persona_corner_3_specialist,persona_defensive_anchor,persona_defensive_rebounder,persona_efficient_scorer,...,persona_playmaker,persona_plus_minus_driver,persona_rim_protector,persona_self_creator,persona_slasher,persona_steal_artist,persona_stretch_big,persona_three_point_specialist,persona_turnover_prone,persona_volume_shooter
0,A'ja Wilson,LVA,2020,1,1,0,0,1,1,0,...,0,0,1,1,0,0,0,0,0,1
1,A'ja Wilson,LVA,2021,1,1,1,0,1,0,0,...,0,0,1,0,0,0,0,0,0,1
2,A'ja Wilson,LVA,2022,1,1,0,0,1,1,0,...,0,1,1,1,0,0,0,0,0,1
3,A'ja Wilson,LVA,2023,1,1,1,0,1,1,1,...,0,1,1,1,0,0,0,0,0,1
4,Aari McDonald,ATL,2021,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,Tyasha Harris,DAL,2020,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
300,Veronica Burton,DAL,2022,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
301,Victaria Saxton,IND,2023,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
302,Victoria Vivians,IND,2020,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [28]:
player_df

,player,year,age,team,home_away,opp,win_margin,gs,mp,gmsc,day,month
0,Tianna Hawkins,2020,29.4,WAS,home,IND,25.0,0,2.0,-0.7,25,7
1,Tianna Hawkins,2020,29.4,WAS,home,LVA,-6.0,0,16.2,3.4,5,8
2,Tianna Hawkins,2020,29.4,WAS,home,NYL,-8.0,0,17.2,-2.7,7,8
3,Tianna Hawkins,2020,29.4,WAS,home,IND,-7.0,0,22.0,15.5,9,8
4,Tianna Hawkins,2020,29.4,WAS,home,MIN,-20.0,1,24.1,10.8,11,8
...,...,...,...,...,...,...,...,...,...,...,...,...
7101,Rennia Davis,2022,23.4,IND,home,LVA,-21.0,0,3.9,0.3,29,7
7102,Rennia Davis,2022,23.4,IND,home,LVA,-25.0,0,10.0,1.5,31,7
7103,Rennia Davis,2022,23.4,IND,away,ATL,-10.0,0,4.3,-1.0,3,8
7104,Rennia Davis,2022,23.5,IND,home,WAS,-12.0,0,7.2,2.7,12,8


In [45]:
# merge top_10_personas into player_df on player, year, and team
player_df = player_df.merge(top_10_personas, on=['player', 'year', 'team'], how='left')

In [46]:
player_df.shape

(7106, 39)

In [74]:
player_df.head()

,player,year,age,team,home_away,opp,win_margin,gs,mp,gmsc,...,persona_playmaker,persona_plus_minus_driver,persona_rim_protector,persona_self_creator,persona_slasher,persona_steal_artist,persona_stretch_big,persona_three_point_specialist,persona_turnover_prone,persona_volume_shooter
0,Tianna Hawkins,2020,29.4,WAS,home,IND,25.0,0,2.0,-0.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Tianna Hawkins,2020,29.4,WAS,home,LVA,-6.0,0,16.2,3.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Tianna Hawkins,2020,29.4,WAS,home,NYL,-8.0,0,17.2,-2.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Tianna Hawkins,2020,29.4,WAS,home,IND,-7.0,0,22.0,15.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Tianna Hawkins,2020,29.4,WAS,home,MIN,-20.0,1,24.1,10.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
# load the persona data CSV from the data folder
persona_2024_data = pd.read_csv("LHL-final-final-project/data/2024_persona_and_team_data.csv")

In [48]:
# load the team data CSV from the data folder
team_df = pd.read_csv("LHL-final-final-project/data/cleaned_full_team_df.csv")

In [49]:
team_df.head()

,team,team_per_game_fg,team_per_game_fga,team_per_game_fg%,team_per_game_3p,team_per_game_3pa,team_per_game_3p%,team_per_game_2p,team_per_game_2pa,team_per_game_2p%,...,team_3par,team_ts%,team_efg%,team_tov%,team_orb%,team_ft/fga,team_efg%.1,team_tov%.1,team_drb%,team_ft/fga.1
0,Atlanta Dream,31.4,71.0,0.442,5.9,16.9,0.350,25.5,54.2,0.471,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
1,Atlanta Dream,30.6,73.5,0.417,6.1,19.8,0.310,24.5,53.7,0.456,...,0.269,0.489,0.459,12.1,24.6,0.154,0.520,16.8,73.7,0.234
2,Atlanta Dream,28.8,68.4,0.420,7.5,21.4,0.351,21.2,47.0,0.452,...,0.313,0.516,0.475,16.7,24.5,0.197,0.494,15.7,78.5,0.240
3,Atlanta Dream,29.4,68.7,0.428,6.4,19.2,0.336,23.0,49.5,0.464,...,0.279,0.527,0.475,14.8,22.4,0.252,0.482,14.2,78.2,0.251
4,Chicago Sky,33.5,68.2,0.491,7.6,21.6,0.351,25.9,46.5,0.557,...,0.317,0.580,0.547,16.6,23.3,0.178,0.503,15.0,76.4,0.205


In [50]:
# Team codes remain the same across years
teams = ['ATL', 'CHI', 'CON', 'DAL', 'IND', 'LAS', 'MIN', 'NYL', 'PHO', 'SEA', 'WAS', 'LVA']

# Base URL with placeholder for team and year
base_url = "https://www.basketball-reference.com/wnba/teams/{team}/{year}/gamelog/"

# Empty list to store DataFrames
frames = []

# Loop through each year and team
for year in range(2020, 2024):
    for team in teams:
        url = base_url.format(team=team, year=year)
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers)
        time.sleep(5)  # Avoid hitting rate limits

        soup = BeautifulSoup(response.content, "html.parser")
        table = soup.find("table", id="wnba_tgl_basic")

        if table is not None:
            df_team = pd.read_html(StringIO(str(table)))[0]

            # Combine multi-level headers
            df_team.columns = [f"{a}_{b}" for a, b in df_team.columns]

            # Drop extra unnamed columns
            drop_cols = [
                'Unnamed: 6_level_0_Unnamed: 6_level_1',
                'Unnamed: 9_level_0_Unnamed: 9_level_1',
                'Unnamed: 26_level_0_Unnamed: 26_level_1'
            ]
            df_team = df_team.drop(columns=drop_cols, errors='ignore')

            # Rename key columns
            df_team = df_team.rename(columns={
                'Unnamed: 0_level_0_Rk': 'Rk',
                'Unnamed: 1_level_0_G#': 'G#',
                'Unnamed: 2_level_0_Date': 'Date',
                'Unnamed: 4_level_0_Opp': 'Opp',
                'Unnamed: 5_level_0_W/L': 'W/L',
                'Unnamed: 7_level_0_Tm': 'Team_Score',
                'Unnamed: 8_level_0_Opp': 'Opp_Score'
            })

            # Add team and year columns at the beginning
            df_team.insert(0, "Year", year)
            df_team.insert(0, "Team", team)

            frames.append(df_team)
        else:
            print(f"No table found for {team}, {year}")

# Combine all collected DataFrames into team_gamelog_df
team_gamelog_df = pd.concat(frames, ignore_index=True)

In [51]:
team_gamelog_df.head()

,Team,Year,Rk,G#,Date,Unnamed: 3_level_0_Unnamed: 3_level_1,Opp,W/L,Team_Score,Opp_Score,...,Opponent_FT,Opponent_FTA,Opponent_FT%,Opponent_ORB,Opponent_TRB,Opponent_AST,Opponent_STL,Opponent_BLK,Opponent_TOV,Opponent_PF
0,ATL,2020,1,1,2020-07-26,NaN,DAL,W,105,95,...,14,20,.700,8,29,18,10,1,13,27
1,ATL,2020,2,2,2020-07-29,@,LVA,L,70,100,...,18,26,.692,14,47,16,11,1,19,14
2,ATL,2020,3,3,2020-07-31,NaN,NYL,W,84,78,...,15,17,.882,8,33,13,8,11,15,23
3,ATL,2020,4,4,2020-08-02,@,IND,L,77,93,...,17,21,.810,9,32,26,5,5,15,13
4,ATL,2020,5,5,2020-08-04,NaN,PHO,L,74,81,...,19,22,.864,10,33,19,7,2,10,18


In [52]:
team_gamelog_df.shape

(1656, 42)

In [53]:
for col in team_gamelog_df.columns:
  print(col)

Team
Year
Rk
G#
Date
Unnamed: 3_level_0_Unnamed: 3_level_1
Opp
W/L
Team_Score
Opp_Score
Team_FG
Team_FGA
Team_FG%
Team_3P
Team_3PA
Team_3P%
Team_FT
Team_FTA
Team_FT%
Team_ORB
Team_TRB
Team_AST
Team_STL
Team_BLK
Team_TOV
Team_PF
Opponent_FG
Opponent_FGA
Opponent_FG%
Opponent_3P
Opponent_3PA
Opponent_3P%
Opponent_FT
Opponent_FTA
Opponent_FT%
Opponent_ORB
Opponent_TRB
Opponent_AST
Opponent_STL
Opponent_BLK
Opponent_TOV
Opponent_PF


In [54]:
# Drop 'Rk' and 'G#' columns
team_gamelog_df = team_gamelog_df.drop(columns=['Rk', 'G#'])

In [56]:
# Remove rows where 'Date' is the header string "Date"
team_gamelog_df = team_gamelog_df[team_gamelog_df['Date'] != 'Date']

In [57]:
# Convert 'Date' to datetime first
team_gamelog_df['Date'] = pd.to_datetime(team_gamelog_df['Date'])

# Create 'day' and 'month' columns
team_gamelog_df['day'] = team_gamelog_df['Date'].dt.day
team_gamelog_df['month'] = team_gamelog_df['Date'].dt.month

# Drop the original 'Date' column
team_gamelog_df = team_gamelog_df.drop(columns=['Date'])

In [58]:
# Rename specified column
team_gamelog_df = team_gamelog_df.rename(columns={"Unnamed: 3_level_0_Unnamed: 3_level_1": "home_away"})

# Lowercase all column names
team_gamelog_df.columns = team_gamelog_df.columns.str.lower()

In [59]:
for col in team_gamelog_df.columns:
  print(col)

team
year
home_away
opp
w/l
team_score
opp_score
team_fg
team_fga
team_fg%
team_3p
team_3pa
team_3p%
team_ft
team_fta
team_ft%
team_orb
team_trb
team_ast
team_stl
team_blk
team_tov
team_pf
opponent_fg
opponent_fga
opponent_fg%
opponent_3p
opponent_3pa
opponent_3p%
opponent_ft
opponent_fta
opponent_ft%
opponent_orb
opponent_trb
opponent_ast
opponent_stl
opponent_blk
opponent_tov
opponent_pf
day
month


In [60]:
# Rename 'w/l' to 'win_loss'
team_gamelog_df = team_gamelog_df.rename(columns={'w/l': 'win_loss'})

# Replace '%' with '_pct' in all column names
team_gamelog_df.columns = team_gamelog_df.columns.str.replace('%', '_pct', regex=False)

In [61]:
for col in team_gamelog_df.columns:
  print(col)

team
year
home_away
opp
win_loss
team_score
opp_score
team_fg
team_fga
team_fg_pct
team_3p
team_3pa
team_3p_pct
team_ft
team_fta
team_ft_pct
team_orb
team_trb
team_ast
team_stl
team_blk
team_tov
team_pf
opponent_fg
opponent_fga
opponent_fg_pct
opponent_3p
opponent_3pa
opponent_3p_pct
opponent_ft
opponent_fta
opponent_ft_pct
opponent_orb
opponent_trb
opponent_ast
opponent_stl
opponent_blk
opponent_tov
opponent_pf
day
month


In [62]:
# save to CSV
team_gamelog_df.to_csv("2020-2023_basketball_reference_gamelog.csv", index=False)

# download to local machine
from google.colab import files
files.download("2020-2023_basketball_reference_gamelog.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [63]:
persona_2024_data

,team,opp,month,day,team_score,opp_score,elite_scorer,efficient_scorer,volume_shooter,three_point_specialist,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,ATL,LAS,5,15,92,81,0,0,1,1,...,0.473,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,ATL,PHO,5,18,85,88,0,0,1,1,...,0.473,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357
2,ATL,DAL,5,21,83,78,0,0,1,1,...,0.473,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357
3,ATL,MIN,5,26,79,92,0,0,1,1,...,0.473,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357
4,ATL,WAS,5,29,73,67,0,0,1,1,...,0.473,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,WAS,CHI,9,11,89,58,0,1,0,0,...,0.504,0.699,0.443,0.439,0.377,0.333,0.612,0.887,0.228,0.347
476,WAS,ATL,9,13,72,69,0,1,0,0,...,0.504,0.699,0.443,0.439,0.377,0.333,0.612,0.887,0.228,0.347
477,WAS,ATL,9,15,73,76,0,1,0,0,...,0.504,0.699,0.443,0.439,0.377,0.333,0.612,0.887,0.228,0.347
478,WAS,NYL,9,17,71,87,0,1,0,0,...,0.504,0.699,0.443,0.439,0.377,0.333,0.612,0.887,0.228,0.347


In [64]:
# Columns to drop
cols_to_drop = [
    'team_fg', 'team_fga', 'team_fg_pct', 'team_3p', 'team_3pa', 'team_3p_pct',
    'team_ft', 'team_fta', 'team_ft_pct', 'team_orb', 'team_trb', 'team_ast',
    'team_stl', 'team_blk', 'team_tov', 'team_pf', 'opponent_fg', 'opponent_fga',
    'opponent_fg_pct', 'opponent_3p', 'opponent_3pa', 'opponent_3p_pct', 'opponent_ft',
    'opponent_fta', 'opponent_ft_pct', 'opponent_orb', 'opponent_trb', 'opponent_ast',
    'opponent_stl', 'opponent_blk', 'opponent_tov', 'opponent_pf', 'win_loss'
]

# Drop specified columns
team_gamelog_df = team_gamelog_df.drop(columns=cols_to_drop)

In [65]:
# Select persona columns (assuming they're all numeric/binary)
persona_cols = player_df.columns.difference(['player', 'team', 'year'])

# Aggregate personas by team-year (sum gives number of persona occurrences per team)
team_persona_df = player_df.groupby(['team', 'year'], as_index=False)[persona_cols].sum()

In [66]:
team_gamelog_df = team_gamelog_df.merge(team_persona_df, on=['team', 'year'], how='left')

In [75]:
team_gamelog_df.head()

,team,year,home_away_x,opp_x,team_score,opp_score,day_x,month_x,age,day_y,...,persona_plus_minus_driver,persona_rim_protector,persona_self_creator,persona_slasher,persona_steal_artist,persona_stretch_big,persona_three_point_specialist,persona_turnover_prone,persona_volume_shooter,win_margin
0,ATL,2020,NaN,DAL,105,95,26.0,7.0,2718.8,1464,...,0.0,0.0,16.0,0.0,0.0,0.0,0.0,16.0,16.0,-767.0
1,ATL,2020,@,LVA,70,100,29.0,7.0,2718.8,1464,...,0.0,0.0,16.0,0.0,0.0,0.0,0.0,16.0,16.0,-767.0
2,ATL,2020,NaN,NYL,84,78,31.0,7.0,2718.8,1464,...,0.0,0.0,16.0,0.0,0.0,0.0,0.0,16.0,16.0,-767.0
3,ATL,2020,@,IND,77,93,2.0,8.0,2718.8,1464,...,0.0,0.0,16.0,0.0,0.0,0.0,0.0,16.0,16.0,-767.0
4,ATL,2020,NaN,PHO,74,81,4.0,8.0,2718.8,1464,...,0.0,0.0,16.0,0.0,0.0,0.0,0.0,16.0,16.0,-767.0


In [67]:
team_gamelog_df.shape

(1608, 44)

In [68]:
for col in team_gamelog_df.columns:
  print(col)

team
year
home_away_x
opp_x
team_score
opp_score
day_x
month_x
age
day_y
gmsc
gs
home_away_y
month_y
mp
opp_y
persona_all_around_star
persona_and_one_machine
persona_catch_and_shoot
persona_corner_3_specialist
persona_defensive_anchor
persona_defensive_rebounder
persona_efficient_scorer
persona_elite_scorer
persona_fast_break_threat
persona_floor_general
persona_free_throw_generator
persona_glass_cleaner
persona_heave_chucker
persona_impact_bench
persona_midrange_sniper
persona_offensive_hub
persona_offensive_rebounder
persona_playmaker
persona_plus_minus_driver
persona_rim_protector
persona_self_creator
persona_slasher
persona_steal_artist
persona_stretch_big
persona_three_point_specialist
persona_turnover_prone
persona_volume_shooter
win_margin


In [76]:
# list of persona columns
persona_cols = [
    'persona_all_around_star', 'persona_and_one_machine', 'persona_catch_and_shoot',
    'persona_corner_3_specialist', 'persona_defensive_anchor', 'persona_defensive_rebounder',
    'persona_efficient_scorer', 'persona_elite_scorer', 'persona_fast_break_threat',
    'persona_floor_general', 'persona_free_throw_generator', 'persona_glass_cleaner',
    'persona_heave_chucker', 'persona_impact_bench', 'persona_midrange_sniper',
    'persona_offensive_hub', 'persona_offensive_rebounder', 'persona_playmaker',
    'persona_plus_minus_driver', 'persona_rim_protector', 'persona_self_creator',
    'persona_slasher', 'persona_steal_artist', 'persona_stretch_big',
    'persona_three_point_specialist', 'persona_turnover_prone', 'persona_volume_shooter'
]

# sum of each persona column
team_gamelog_df[persona_cols].sum()

,0
persona_all_around_star,6006.0
persona_and_one_machine,2884.0
persona_catch_and_shoot,8572.0
persona_corner_3_specialist,9396.0
persona_defensive_anchor,2553.0
persona_defensive_rebounder,5121.0
persona_efficient_scorer,2614.0
persona_elite_scorer,3327.0
persona_fast_break_threat,5438.0
persona_floor_general,4510.0


In [77]:
# Display columns with null counts, sorted descending, showing only columns with nulls
team_gamelog_df.isnull().sum()[team_gamelog_df.isnull().sum() > 0].sort_values(ascending=False)

,0
home_away_x,828
opp_x,48
team_score,48
opp_score,48
day_x,48
month_x,48


In [78]:
# Drop columns ending with '_y' and 'win_margin'
cols_to_drop = [col for col in team_gamelog_df.columns if col.endswith('_y')] + ['win_margin']
team_gamelog_df = team_gamelog_df.drop(columns=cols_to_drop)

# Remove '_x' suffix from column names
team_gamelog_df.columns = [col[:-2] if col.endswith('_x') else col for col in team_gamelog_df.columns]

# Update home_away column ('@' to 0, NaN to 1)
team_gamelog_df['home_away'] = team_gamelog_df['home_away'].replace({'@': 0}).fillna(1)

<ipython-input-78-5507a1d9b49b>:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  team_gamelog_df['home_away'] = team_gamelog_df['home_away'].replace({'@': 0}).fillna(1)


In [80]:
for col in team_gamelog_df.columns:
  print(col)

team
year
home_away
opp
team_score
opp_score
day
month
age
gmsc
gs
mp
persona_all_around_star
persona_and_one_machine
persona_catch_and_shoot
persona_corner_3_specialist
persona_defensive_anchor
persona_defensive_rebounder
persona_efficient_scorer
persona_elite_scorer
persona_fast_break_threat
persona_floor_general
persona_free_throw_generator
persona_glass_cleaner
persona_heave_chucker
persona_impact_bench
persona_midrange_sniper
persona_offensive_hub
persona_offensive_rebounder
persona_playmaker
persona_plus_minus_driver
persona_rim_protector
persona_self_creator
persona_slasher
persona_steal_artist
persona_stretch_big
persona_three_point_specialist
persona_turnover_prone
persona_volume_shooter


In [82]:
# List of columns to drop
cols_to_drop = [
    'age', 'gmsc', 'gs', 'mp',
    'persona_all_around_star', 'persona_and_one_machine', 'persona_catch_and_shoot',
    'persona_corner_3_specialist', 'persona_defensive_anchor', 'persona_defensive_rebounder',
    'persona_efficient_scorer', 'persona_elite_scorer', 'persona_fast_break_threat',
    'persona_floor_general', 'persona_free_throw_generator', 'persona_glass_cleaner',
    'persona_heave_chucker', 'persona_impact_bench', 'persona_midrange_sniper',
    'persona_offensive_hub', 'persona_offensive_rebounder', 'persona_playmaker',
    'persona_plus_minus_driver', 'persona_rim_protector', 'persona_self_creator',
    'persona_slasher', 'persona_steal_artist', 'persona_stretch_big',
    'persona_three_point_specialist', 'persona_turnover_prone', 'persona_volume_shooter'
]

# Drop the columns from team_gamelog_df
team_gamelog_df = team_gamelog_df.drop(columns=cols_to_drop, errors='ignore')

In [84]:
team_gamelog_df.shape

(1608, 8)

In [85]:
player_df.head()

,player,year,age,team,home_away,opp,win_margin,gs,mp,gmsc,...,persona_playmaker,persona_plus_minus_driver,persona_rim_protector,persona_self_creator,persona_slasher,persona_steal_artist,persona_stretch_big,persona_three_point_specialist,persona_turnover_prone,persona_volume_shooter
0,Tianna Hawkins,2020,29.4,WAS,home,IND,25.0,0,2.0,-0.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Tianna Hawkins,2020,29.4,WAS,home,LVA,-6.0,0,16.2,3.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Tianna Hawkins,2020,29.4,WAS,home,NYL,-8.0,0,17.2,-2.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Tianna Hawkins,2020,29.4,WAS,home,IND,-7.0,0,22.0,15.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Tianna Hawkins,2020,29.4,WAS,home,MIN,-20.0,1,24.1,10.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [86]:
# Check if any rows have persona_playmaker > 1
(player_df['persona_playmaker'] > 1).sum()

np.int64(0)

In [87]:
player_df['persona_playmaker'].value_counts()

,count
persona_playmaker,
0.0,1234
1.0,166


In [88]:
# Columns to group by for a single game/team
group_cols = ['year', 'month', 'day', 'team']

# Persona columns (all columns starting with 'persona_')
persona_cols = [col for col in player_df.columns if col.startswith('persona_')]

# Aggregate personas at the game/team level
game_personas_df = player_df.groupby(group_cols, as_index=False)[persona_cols].sum()

In [89]:
game_personas_df.shape

(1560, 31)

In [90]:
game_personas_df.head()

,year,month,day,team,persona_all_around_star,persona_and_one_machine,persona_catch_and_shoot,persona_corner_3_specialist,persona_defensive_anchor,persona_defensive_rebounder,...,persona_playmaker,persona_plus_minus_driver,persona_rim_protector,persona_self_creator,persona_slasher,persona_steal_artist,persona_stretch_big,persona_three_point_specialist,persona_turnover_prone,persona_volume_shooter
0,2020,7,25,IND,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2020,7,25,LAS,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2020,7,25,NYL,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0
3,2020,7,25,PHO,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2020,7,25,SEA,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [91]:
# Check games without any matching player personas
no_persona_games = team_gamelog_df.merge(
    game_personas_df,
    on=['year', 'month', 'day', 'team'],
    how='left',
    indicator=True
).query("_merge == 'left_only'")

len(no_persona_games)


48

In [92]:
# Identify persona columns explicitly
persona_cols = [col for col in game_personas_df.columns if col.startswith('persona_')]

# Merge only persona columns into team_gamelog_df
team_gamelog_df = team_gamelog_df.merge(
    game_personas_df[['year', 'month', 'day', 'team'] + persona_cols],
    on=['year', 'month', 'day', 'team'],
    how='left'
)

In [93]:
team_gamelog_df.shape

(1608, 35)

In [94]:
team_gamelog_df.head()

,team,year,home_away,opp,team_score,opp_score,day,month,persona_all_around_star,persona_and_one_machine,...,persona_playmaker,persona_plus_minus_driver,persona_rim_protector,persona_self_creator,persona_slasher,persona_steal_artist,persona_stretch_big,persona_three_point_specialist,persona_turnover_prone,persona_volume_shooter
0,ATL,2020,1.0,DAL,105,95,26.0,7.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
1,ATL,2020,0.0,LVA,70,100,29.0,7.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
2,ATL,2020,1.0,NYL,84,78,31.0,7.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
3,ATL,2020,0.0,IND,77,93,2.0,8.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
4,ATL,2020,1.0,PHO,74,81,4.0,8.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0


In [96]:
# Remove 'persona_' prefix from column names
team_gamelog_df.columns = [col.replace('persona_', '') if col.startswith('persona_') else col
                           for col in team_gamelog_df.columns]

In [97]:
for col in team_gamelog_df.columns:
  print(col)

team
year
home_away
opp
team_score
opp_score
day
month
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_protector
self_creator
slasher
steal_artist
stretch_big
three_point_specialist
turnover_prone
volume_shooter


In [99]:
# Show rows where 'opp' is null
team_gamelog_df[team_gamelog_df['opp'].isnull()]

,team,year,home_away,opp,team_score,opp_score,day,month,all_around_star,and_one_machine,...,playmaker,plus_minus_driver,rim_protector,self_creator,slasher,steal_artist,stretch_big,three_point_specialist,turnover_prone,volume_shooter
20,ATL,2020,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,CHI,2020,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,CON,2020,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89,DAL,2020,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
112,IND,2020,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
135,LAS,2020,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158,MIN,2020,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,NYL,2020,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
204,PHO,2020,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
227,SEA,2020,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
# Drop rows where 'opp' is null
team_gamelog_df = team_gamelog_df.dropna(subset=['opp'])

In [101]:
# Display columns with null counts, sorted descending, showing only columns with nulls
team_gamelog_df.isnull().sum()[team_gamelog_df.isnull().sum() > 0].sort_values(ascending=False)

,0


In [103]:
team_df.head()

,team,team_per_game_fg,team_per_game_fga,team_per_game_fg%,team_per_game_3p,team_per_game_3pa,team_per_game_3p%,team_per_game_2p,team_per_game_2pa,team_per_game_2p%,...,team_3par,team_ts%,team_efg%,team_tov%,team_orb%,team_ft/fga,team_efg%.1,team_tov%.1,team_drb%,team_ft/fga.1
0,Atlanta Dream,31.4,71.0,0.442,5.9,16.9,0.350,25.5,54.2,0.471,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
1,Atlanta Dream,30.6,73.5,0.417,6.1,19.8,0.310,24.5,53.7,0.456,...,0.269,0.489,0.459,12.1,24.6,0.154,0.520,16.8,73.7,0.234
2,Atlanta Dream,28.8,68.4,0.420,7.5,21.4,0.351,21.2,47.0,0.452,...,0.313,0.516,0.475,16.7,24.5,0.197,0.494,15.7,78.5,0.240
3,Atlanta Dream,29.4,68.7,0.428,6.4,19.2,0.336,23.0,49.5,0.464,...,0.279,0.527,0.475,14.8,22.4,0.252,0.482,14.2,78.2,0.251
4,Chicago Sky,33.5,68.2,0.491,7.6,21.6,0.351,25.9,46.5,0.557,...,0.317,0.580,0.547,16.6,23.3,0.178,0.503,15.0,76.4,0.205


In [104]:
for col in team_df.columns:
  print(col)

team
team_per_game_fg
team_per_game_fga
team_per_game_fg%
team_per_game_3p
team_per_game_3pa
team_per_game_3p%
team_per_game_2p
team_per_game_2pa
team_per_game_2p%
team_per_game_ft
team_per_game_fta
team_per_game_ft%
team_per_game_orb
team_per_game_drb
team_per_game_trb
team_per_game_ast
team_per_game_stl
team_per_game_blk
team_per_game_tov
team_per_game_pf
team_per_game_pts
year
opp_per_game_fg
opp_per_game_fga
opp_per_game_fg%
opp_per_game_3p
opp_per_game_3pa
opp_per_game_3p%
opp_per_game_2p
opp_per_game_2pa
opp_per_game_2p%
opp_per_game_ft
opp_per_game_fta
opp_per_game_ft%
opp_per_game_orb
opp_per_game_drb
opp_per_game_trb
opp_per_game_ast
opp_per_game_stl
opp_per_game_blk
opp_per_game_tov
opp_per_game_pf
opp_per_game_pts
team_per_poss_fg
team_per_poss_fga
team_per_poss_fg%
team_per_poss_3p
team_per_poss_3pa
team_per_poss_3p%
team_per_poss_2p
team_per_poss_2pa
team_per_poss_2p%
team_per_poss_ft
team_per_poss_fta
team_per_poss_ft%
team_per_poss_orb
team_per_poss_drb
team_per_poss_trb

In [105]:
# Merge team_df into team_gamelog_df on 'team' and 'year' with default suffixes
persona_and_team_df = team_gamelog_df.merge(
    team_df,
    on=['team', 'year'],
    how='left'
)

In [106]:
persona_and_team_df.head()

,team,year,home_away,opp,team_score,opp_score,day,month,all_around_star,and_one_machine,...,team_3par,team_ts%,team_efg%,team_tov%,team_orb%,team_ft/fga,team_efg%.1,team_tov%.1,team_drb%,team_ft/fga.1
0,ATL,2020,1.0,DAL,105,95,26.0,7.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ATL,2020,0.0,LVA,70,100,29.0,7.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ATL,2020,1.0,NYL,84,78,31.0,7.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ATL,2020,0.0,IND,77,93,2.0,8.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ATL,2020,1.0,PHO,74,81,4.0,8.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [107]:
persona_and_team_df.shape

(1560, 174)

In [108]:
for col in persona_and_team_df.columns:
  print(col)

team
year
home_away
opp
team_score
opp_score
day
month
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_protector
self_creator
slasher
steal_artist
stretch_big
three_point_specialist
turnover_prone
volume_shooter
team_per_game_fg
team_per_game_fga
team_per_game_fg%
team_per_game_3p
team_per_game_3pa
team_per_game_3p%
team_per_game_2p
team_per_game_2pa
team_per_game_2p%
team_per_game_ft
team_per_game_fta
team_per_game_ft%
team_per_game_orb
team_per_game_drb
team_per_game_trb
team_per_game_ast
team_per_game_stl
team_per_game_blk
team_per_game_tov
team_per_game_pf
team_per_game_pts
opp_per_game_fg
opp_per_game_fga
opp_per_game_fg%
opp_per_game_3p
opp_per_game_3pa
opp_per_game_3p%
opp_per_game_2p
opp_per_game_2pa
opp_per_game_2p%
op

In [109]:
# save to CSV
persona_and_team_df.to_csv("2020-2023_persona_and_team_data.csv", index=False)

# download to local machine
from google.colab import files
files.download("2020-2023_persona_and_team_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [112]:
# Make column titles lowercase
persona_2024_data.columns = persona_2024_data.columns.str.lower()

In [115]:
# List of columns to drop
cols_to_drop = [
    'opp_totals_fg', 'opp_totals_fga', 'opp_totals_fg%', 'opp_totals_3p', 'opp_totals_3pa',
    'opp_totals_3p%', 'opp_totals_2p', 'opp_totals_2pa', 'opp_totals_2p%', 'opp_totals_ft',
    'opp_totals_fta', 'opp_totals_ft%', 'opp_totals_orb', 'opp_totals_drb', 'opp_totals_trb',
    'opp_totals_ast', 'opp_totals_stl', 'opp_totals_blk', 'opp_totals_tov', 'opp_totals_pf',
    'opp_totals_pts', 'team_totals_fg', 'team_totals_fga', 'team_totals_fg%', 'team_totals_3p',
    'team_totals_3pa', 'team_totals_3p%', 'team_totals_2p', 'team_totals_2pa', 'team_totals_2p%',
    'team_totals_ft', 'team_totals_fta', 'team_totals_ft%', 'team_totals_orb', 'team_totals_drb',
    'team_totals_trb', 'team_totals_ast', 'team_totals_stl', 'team_totals_blk', 'team_totals_tov',
    'team_totals_pf', 'team_totals_pts', 'team_w', 'team_l', 'team_pw', 'team_pl'
]

# Drop the columns
persona_2024_data = persona_2024_data.drop(columns=cols_to_drop, errors='ignore')

In [119]:
persona_2024_data.columns = persona_2024_data.columns.str.replace('_shooting', '', regex=False)

In [121]:
persona_2024_data['year'] = 2024

In [122]:
# Find columns in persona_and_team_df not in persona_2024_data
cols_only_in_persona_and_team_df = set(persona_and_team_df.columns) - set(persona_2024_data.columns)

# Find columns in persona_2024_data not in persona_and_team_df
cols_only_in_persona_2024_data = set(persona_2024_data.columns) - set(persona_and_team_df.columns)

print("Columns only in persona_and_team_df:")
print(cols_only_in_persona_and_team_df)

print("\nColumns only in persona_2024_data:")
print(cols_only_in_persona_2024_data)

Columns only in persona_and_team_df:
{'home_away'}

Columns only in persona_2024_data:
set()


In [114]:
for col in persona_2024_data.columns:
  print(col)

team
opp
month
day
team_score
opp_score
elite_scorer
efficient_scorer
volume_shooter
three_point_specialist
slasher
free_throw_generator
and_one_machine
playmaker
offensive_hub
turnover_prone
floor_general
plus_minus_driver
self_creator
rim_protector
steal_artist
defensive_anchor
glass_cleaner
offensive_rebounder
defensive_rebounder
midrange_sniper
corner_3_specialist
catch_and_shoot
stretch_big
heave_chucker
all_around_star
impact_bench
fast_break_threat
team_per_game_fg
team_per_game_fga
team_per_game_fg%
team_per_game_3p
team_per_game_3pa
team_per_game_3p%
team_per_game_2p
team_per_game_2pa
team_per_game_2p%
team_per_game_ft
team_per_game_fta
team_per_game_ft%
team_per_game_orb
team_per_game_drb
team_per_game_trb
team_per_game_ast
team_per_game_stl
team_per_game_blk
team_per_game_tov
team_per_game_pf
team_per_game_pts
team_totals_fg
team_totals_fga
team_totals_fg%
team_totals_3p
team_totals_3pa
team_totals_3p%
team_totals_2p
team_totals_2pa
team_totals_2p%
team_totals_ft
team_totals

In [123]:
# load the team data CSV from the data folder
home_away_2024_df = pd.read_csv("LHL-final-final-project/data/2024_basketball_reference_gamelog.csv")

In [124]:
home_away_2024_df.head()

,team,g_num,year,month,day,home_away,opp,win_loss,team_score,opp_score,...,opponent_ft,opponent_fta,opponent_ft_pct,opponent_orb,opponent_trb,opponent_ast,opponent_stl,opponent_blk,opponent_tov,opponent_pf
0,ATL,1,2024,5,15,2,LAS,1,92,81,...,13,22,0.591,8,35,24,8,2,11,17
1,ATL,2,2024,5,18,2,PHO,2,85,88,...,28,32,0.875,7,38,17,10,5,12,22
2,ATL,3,2024,5,21,1,DAL,1,83,78,...,16,20,0.800,11,35,17,6,6,14,20
3,ATL,4,2024,5,26,1,MIN,2,79,92,...,17,23,0.739,5,31,23,7,4,10,17
4,ATL,5,2024,5,29,2,WAS,1,73,67,...,7,7,1.000,9,31,21,6,3,13,17


In [125]:
# Keep only the specified columns
home_away_2024_df = home_away_2024_df[['team', 'month', 'day', 'home_away']]

# Replace all 2 values in 'home_away' with 0
home_away_2024_df['home_away'] = home_away_2024_df['home_away'].replace(2, 0)

<ipython-input-125-4c00e2712c7d>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  home_away_2024_df['home_away'] = home_away_2024_df['home_away'].replace(2, 0)


In [128]:
home_away_2024_df.head()

,team,month,day,home_away
0,ATL,5,15,0
1,ATL,5,18,0
2,ATL,5,21,1
3,ATL,5,26,1
4,ATL,5,29,0


In [129]:
persona_2024_data = persona_2024_data.merge(
    home_away_2024_df[['team', 'day', 'month', 'home_away']],
    on=['team', 'day', 'month'],
    how='left'
)

In [130]:
# Find columns in persona_and_team_df not in persona_2024_data
cols_only_in_persona_and_team_df = set(persona_and_team_df.columns) - set(persona_2024_data.columns)

# Find columns in persona_2024_data not in persona_and_team_df
cols_only_in_persona_2024_data = set(persona_2024_data.columns) - set(persona_and_team_df.columns)

print("Columns only in persona_and_team_df:")
print(cols_only_in_persona_and_team_df)

print("\nColumns only in persona_2024_data:")
print(cols_only_in_persona_2024_data)

Columns only in persona_and_team_df:
set()

Columns only in persona_2024_data:
set()


In [131]:
# save to CSV
persona_2024_data.to_csv("2024_persona_and_team_data.csv", index=False)

# download to local machine
from google.colab import files
files.download("2024_persona_and_team_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [132]:
# Concatenate (stack) the two DataFrames vertically
persona_and_team_df = pd.concat([persona_and_team_df, persona_2024_data], ignore_index=True)

In [133]:
persona_and_team_df.shape

(2040, 174)

In [134]:
# save to CSV
persona_and_team_df.to_csv("all_persona_and_team_data.csv", index=False)

# download to local machine
from google.colab import files
files.download("all_persona_and_team_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>